# Xcelerate — Inférence de possession : modèles, lois du jeu, fusion et sélection

**More Than a Hack 2027 · FC Barcelona – Athletic Club · Livrable technique de l’équipe Xcelerate · 15 novembre 2026**

> **Objectif.** Déduire automatiquement l'état de chaque frame vidéo : `H` (Barcelona contrôle, ballon en jeu), `A` (Athletic contrôle, ballon en jeu), `D` (ballon hors jeu / arrêt du jeu) et `X` (hors plages à prédire ou abstention). Le format final contient **199 319 lignes**, de `frame=0` à `frame=199318`.

## Architecture expérimentale et hypothèses

| Famille | Modèle | Hypothèse testée | Comparaison principale |
|---|---|---|---|
| 01 | `nearest` | Le joueur le plus proche indique la possession | Baseline naïve |
| 01 | `baseline_hmm` | Scores géométriques + continuité Viterbi | `nearest` |
| 02 | `switch_strong` | Coût élevé de basculement H↔A et entrée/sortie D | `baseline_hmm` |
| 03 | `control_strong` | Un contrôle durable et exclusif vaut mieux que la proximité | `switch_strong` |
| 04 | `control_duel_strong` | Le doute pendant un duel doit favoriser la persistance | `control_strong` |
| 05 | `control_touch_strong` | Les changements de vitesse du ballon apportent un indice de touche | `control_duel_strong` |
| 06 | `control_restart_strong` | Les remises en jeu réinitialisent le contrôleur précédent | `control_touch_strong` (ablation cible) |
| 07 | `graph_geometry` | La densité et la pression collectives apportent du contexte | `baseline_hmm` |
| 08 | `adaptive_multiscale` | Plusieurs échelles évitent la réactivité excessive | `graph_geometry` |
| 09 | `fusion_control_multiscale` | Fusion de preuves de contrôle et de contexte | `control_duel_strong` |
| 10 | `D_V2_strong` | Les sorties, reprises et événements sont plus utiles qu'un biais D | `switch_strong` — **D non validé** |
| 11 | `fusion_D_V2_strong` | Contrôle H/A + arrêt D spécialisé | `fusion_control_multiscale` — **D non validé** |
| 12 | `TabICLv2_fusion` | Un TFM entraîné **uniquement sur des matchs externes** apporte des probabilités H/A | `switch_strong`, boosting — facultatif |

La méthode est construite par **ablations** : à chaque changement, nous gardons la même temporalité autant que possible et isolons l'effet d'un indice. « Meilleur nombre de possessions » ne signifie pas « meilleur modèle ».

**Validation.** Cinq matchs SkillCorner distincts pour la comparaison H/A et un match final réservé. Les périodes SkillCorner non annotées **ne sont pas des exemples `D`**. La GT publique de cinq minutes du concours sert **uniquement** à vérifier le pipeline après la sélection, jamais à entraîner, ajuster ou départager les modèles. Aucune annotation manuelle du match cible. Le code et les modèles doivent être accessibles et capables de régénérer exactement le CSV ; les données privées Metrica ne sont pas publiées.

Sources : [Règles](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/RULES.md) · [Protocole de score](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/EVALUATION.md) · [SkillCorner Open Data](https://github.com/SkillCorner/opendata) · [TabICLv2](https://tabicl.readthedocs.io/en/latest/).

**Reproduction:** The provided match is accessed only through an authorized private Kaggle input (`erickambire/bar-data`); the public repository contains no tracking/video/events exports. GitHub release and report are separate from the competition files.


## 0 · Paramètres fixés avant les expériences

Nous fixons les valeurs par principe physique et à partir des travaux antérieurs, **pas** depuis les cinq minutes publiques. Les hypothèses seront mises en défaut par des évaluations externes. Le match `2011166` a déjà été consulté dans nos anciennes expériences : **ce n'est plus un holdout inédit**. Nous utilisons `2017461` pour une vérification unique après le choix.

Pour TabICLv2, le *screening* sur frames échantillonnées est lancé si le package est disponible ; une intégration à la soumission exige ensuite une **validation temporelle complète sur chacun des cinq matchs**, activable explicitement (coût GPU important).

In [ ]:
import os,sys,json,time,gc,subprocess,importlib.util,warnings,hashlib
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
SEED=2027
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT.mkdir(parents=True,exist_ok=True)
DEV_IDS=[1886347,1899585,1925299,1996435,2006229]
FRESH_HOLDOUT=2017461  # un seul test, APRES verrouillage
PEN_BASE=(13.,10.,7.)
PEN_STRONG=(23.,17.,11.)
PEN_FRONTIER=(14.,10.,8.)
PEN_TEMPERATURE_3=(23.,17.,11.)
R_CTRL=1.8; CTRL_MIN_LEN=5; CTRL_MAX_AGE_SEC=10.
CTRL_MAX_BALL_SPEED=16.  # m/s ; exclure les passes rapides comme nouveaux contrôles
TOUCH_ACCEL=35.          # m/s² ; indice de touche, pas label certain
RESTART_PRIOR_WEIGHT=0.35
RUN_PSEUDO=False         # variante expérimentale, exécution facultative
RUN_TABICL_SCREEN=True   # test macroF1 externe sur sous-échantillon
RUN_TABICL_FULL_CV=False # mettre True pour éligibilité réelle au leaderboard spellF1
TABICL_TRAIN_PER_GAME=900
TABICL_VAL_PER_GAME=1000
TABICL_QUERY_BATCH=1200
RUN_FRESH_HOLDOUT_ONCE=True
RUN_PUBLIC_PIPELINE_CHECK=True
print('Sorties:',OUT,'| dev:',DEV_IDS,'| holdout réservé:',FRESH_HOLDOUT)

## 1 · Ingestion Metrica : tracking, équipes, tags, alignement

Nous utilisons la vidéo tactique **25 FPS**, l'ATD (`frame_id=frame+1`), le XML des équipes et le Smart Tagging. Les données absentes restent `NaN` : un ballon non visible n'est jamais attribué automatiquement à une équipe. Les deux périodes évaluées sont **[11185, 90651]** et **[110808, 190197]** (bornes incluses). Le deuxième engagement est connu à la frame **113742** : les frames antérieures de la deuxième période sont imposées `D`.

Nous n'utilisons ni une transcription manuelle de la vidéo, ni des événements humains du même match. Nous gardons le tagging automatique comme une source incertaine. Les fichiers privés sont lus depuis `BAR-DATA` sans publication.

In [ ]:
# Common, self-contained utilities. This entire source is embedded in each notebook.
import os, sys, time, warnings, json
from pathlib import Path
import xml.etree.ElementTree as ET
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from IPython.display import display

FPS = 25
N_FRAMES = 199319
HALVES = [(11185, 90651), (110808, 190197)]
SECOND_KICKOFF = 113742
LENGTH_M, WIDTH_M = 105.3, 68.0
CODES = np.array(['H','A','D'],dtype='<U1')
OUTPUT_DIR = Path(os.getenv('MTAH_OUTPUT_DIR', '/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data'))
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
RNG = np.random.default_rng(2027)
print('Xcelerate | NumPy:', np.__version__, '| Sorties:', OUTPUT_DIR)

def local_tag(tag): return tag.rsplit('}',1)[-1].split(':')[-1]
def child_text(element, key):
    for el in element:
        if local_tag(el.tag).lower()==key.lower(): return (el.text or '').strip()
    return ''

def locate_assets():
    names={'tracking':'BAR-ATH_tactical_atd.txt','metadata':'BAR-ATH_tactical_atd.xml',
           'tagging':'BAR-ATH_tactical_smart_tagging.xml','homography':'BAR-ATH_tactical_homography.csv',
           'video':'BAR-ATH_tactical.mp4','broadcast':'BAR-ATH_broadcast.mp4'}
    roots=[os.getenv('MTAH_DATA_DIR',''),'/kaggle/input/datasets/erickambire/bar-data','/kaggle/input/bar-data']
    for base in list(filter(None,roots)):
        p=Path(base)
        if (p/names['tracking']).is_file():
            return {k:p/v for k,v in names.items()}
        if (p/'tactical'/names['tracking']).is_file():
            return {k: (p/'tactical'/v if k!='broadcast' else p/'broadcast'/v) for k,v in names.items()}
    root=Path('/kaggle/input')
    hits=list(root.rglob(names['tracking'])) if root.exists() else []
    if hits:
        p=hits[0].parent
        print('Dataset identifié :',p)
        return {k:p/v if k!='broadcast' else p.parent/'broadcast'/v for k,v in names.items()}
    raise FileNotFoundError('Fichier ATD introuvable. Kaggle > Add Input > ajouter BAR-DATA, ou définir MTAH_DATA_DIR.')

def read_metadata(path):
    root=ET.parse(path).getroot()
    teams=[x for x in root.iter() if local_tag(x.tag)=='Team']
    players=[x for x in root.iter() if local_tag(x.tag)=='Player']
    if len(teams)!=2 or not players: raise ValueError('Structure XML des équipes inattendue')
    tid=teams[0].attrib['id']; aid=teams[1].attrib['id']
    codes=np.array([0 if x.attrib.get('teamId')==tid else 1 if x.attrib.get('teamId')==aid else -1 for x in players])
    if np.any(codes<0):raise ValueError('Track slot sans équipe')
    return child_text(teams[0],'Name'),child_text(teams[1],'Name'),codes

def load_atd(path, n_slots, n_frames=N_FRAMES):
    # 199319 x 46 x 3 in float32 ≈110MB, without video loading.
    players=np.full((n_frames,n_slots,3),np.nan,np.float32)
    ball=np.full((n_frames,3),np.nan,np.float32)
    count=0
    with open(path,encoding='utf8') as stream:
        for i,line in enumerate(stream):
            if i>=n_frames:raise ValueError('ATD plus long que la vidéo tactique')
            parts=line.strip().split(':',2)
            if len(parts)!=3:raise ValueError(f'ATD incorrect ligne {i+1}')
            if int(parts[0])!=i+1:raise ValueError(f'Décalage ATD à la ligne {i+1}')
            p=np.fromstring(parts[1].replace(';',','),sep=',',dtype=np.float32)
            b=np.fromstring(parts[2],sep=',',dtype=np.float32)
            if len(p)!=3*n_slots or len(b)!=3:raise ValueError(f'Taille inattendue ligne {i+1}: {len(p)}, {len(b)}')
            players[i]=p.reshape(n_slots,3);ball[i]=b
            count+=1
    if count!=n_frames:raise ValueError(f'{count} lignes ATD, {n_frames} requises')
    return players,ball

def norm(s):
    s=unicodedata.normalize('NFKD',str(s or ''))
    return ''.join(c for c in s if not unicodedata.combining(c)).strip().lower()
def side(name):
    name=norm(name)
    if 'barcelona' in name:return 'H'
    if 'athletic' in name or 'atletic' in name:return 'A'
    return None

def read_tagging(path):
    root=ET.parse(path).getroot();events=[]
    for node in root.iter():
        if local_tag(node.tag).lower()!='instance':continue
        tags={}
        for label in node:
            if local_tag(label.tag).lower()=='label':tags[child_text(label,'group')]=child_text(label,'text')
        events.append(dict(lo=round(float(child_text(node,'start'))*FPS),
                           hi=round(float(child_text(node,'end'))*FPS),
                           code=child_text(node,'code').upper(), side=side(tags.get('Team')),
                           type=tags.get('Type','')))
    return events

ATTACK={'BUILD UP','PROGRESSION','CREATING CHANCES','ATTACKING TRANSITION',
        'LONG BALL','BALL IN FINAL THIRD','BALL IN THE BOX','PLAYERS IN THE BOX'}

def add_tag_features(df,events):
    n=len(df)
    fields=['tag_attack_home','tag_attack_away','tag_setpiece','tag_transition_home',
            'tag_transition_away','tag_defensive_home','tag_defensive_away','tag_count']
    values={k:np.zeros(n,np.int16) for k in fields}
    for ev in events:
        lo=max(0,ev['lo']);hi=min(n,ev['hi']+1)
        if hi<=lo:continue
        values['tag_count'][lo:hi]+=1
        if ev['code']=='SET PIECES':values['tag_setpiece'][lo:hi]+=1
        key='home' if ev['side']=='H' else 'away' if ev['side']=='A' else None
        if key:
            if ev['code'] in ATTACK:values['tag_attack_'+key][lo:hi]+=1
            if ev['code']=='ATTACKING TRANSITION':values['tag_transition_'+key][lo:hi]+=1
            if ev['code'].startswith('DEFENDING '):values['tag_defensive_'+key][lo:hi]+=1
    for k,v in values.items():df[k]=v
    return df

def make_features(players,ball,team_codes):
    n=len(ball);mask=np.isfinite(ball[:,:2]).all(axis=1)
    df=pd.DataFrame({'frame':np.arange(n,dtype=np.int32),'frame_id':np.arange(1,n+1,dtype=np.int32),
                     'ball_visible':mask.astype(np.int8),'ball_x':ball[:,0],
                     'ball_y':ball[:,1],'ball_speed':ball[:,2]})
    for team,key in [(0,'home'),(1,'away')]:
        p=players[:,team_codes==team,:]
        seen=np.isfinite(p[:,:,:2]).all(axis=2)
        speed=np.where(seen,p[:,:,2],np.nan)
        valid_counts=np.isfinite(speed).sum(axis=1)
        sums=np.nansum(speed,axis=1)
        df[key+'_mean_speed']=np.divide(sums,valid_counts,out=np.full(n,np.nan),where=valid_counts>0)
        df[key+'_visible_players']=seen.sum(axis=1)
        dx=(p[:,:,0]-ball[:,None,0])*LENGTH_M
        dy=(p[:,:,1]-ball[:,None,1])*WIDTH_M
        dist=np.sqrt(dx*dx+dy*dy)
        dist[~seen | ~mask[:,None]]=np.inf
        # Extract 2 nearest tracked players; avoid identity assumptions.
        near=np.partition(dist, min(1,dist.shape[1]-1),axis=1)[:,:2]
        one=near[:,0].copy();one[~np.isfinite(one)]=np.nan
        df[key+'_nearest_ball_dist']=one
        two=near[:,1].copy() if near.shape[1]>1 else one.copy()
        two[~np.isfinite(two)]=np.nan
        df[key+'_second_nearest_ball_dist']=two
        for r in (2,5,10):
            count=(dist<r).sum(axis=1).astype(float)
            count[~mask]=np.nan
            df[f'{key}_players_within_{r}m']=count
    df['nearest_dist_diff']=df['home_nearest_ball_dist']-df['away_nearest_ball_dist']
    df['closest_distance']=df[['home_nearest_ball_dist','away_nearest_ball_dist']].min(axis=1)
    # Temporal features, never connect across halves in the model decoder.
    return df

def load_match():
    paths=locate_assets()
    for k in ('tracking','metadata','tagging'):
        if not paths[k].is_file():raise FileNotFoundError(f'Fichier requis absent: {paths[k]}')
    home,away,team=read_metadata(paths['metadata'])
    print('Équipes:',home,'vs',away,'| slots:',len(team))
    p,b=load_atd(paths['tracking'],len(team),N_FRAMES)
    f=make_features(p,b,team)
    e=read_tagging(paths['tagging']);add_tag_features(f,e)
    print(f'{len(f):,} frames, {len(f.columns)} variables, {len(e)} événements (chevauchements conservés).')
    return f,{'players':p,'ball':b,'team':team,'events':e,'paths':paths}

DEFAULT_PARAMS=dict(prox_scale=4.0,prox_clip=2.0,attack_weight=0.4,
                    dead_bias=-0.7,setpiece_weight=0.3,near_distance=2.0,
                    near_penalty=1.2,slow_speed=1.2,slow_bonus=0.5,
                    outside_bonus=2.0,temperature=1.0)

def make_logits(df,parameters=None):
    p={**DEFAULT_PARAMS,**(parameters or {})};n=len(df)
    z=np.zeros((n,3),np.float64)
    get=lambda key: pd.to_numeric(df[key],errors='coerce').to_numpy(dtype=float)
    diff=get('nearest_dist_diff');seen=get('ball_visible')>0.5
    valid=seen & np.isfinite(diff)
    prox=np.zeros(n)
    prox[valid]=np.clip(diff[valid]/max(p['prox_scale'],1e-6),-p['prox_clip'],p['prox_clip'])
    z[:,0]-=prox;z[:,1]+=prox
    z[:,0]+=p['attack_weight']*np.minimum(get('tag_attack_home'),3)
    z[:,1]+=p['attack_weight']*np.minimum(get('tag_attack_away'),3)
    z[:,2]+=p['dead_bias']+p['setpiece_weight']*np.minimum(get('tag_setpiece'),2)
    near=get('closest_distance')
    z[:,2]-=p['near_penalty']*(valid & np.isfinite(near) & (near<=p['near_distance']))
    sp_h=get('home_mean_speed');sp_a=get('away_mean_speed')
    slow=(np.nan_to_num(sp_h,nan=100)<p['slow_speed']) & (np.nan_to_num(sp_a,nan=100)<p['slow_speed'])
    z[:,2]+=p['slow_bonus']*(~seen & slow)
    bx=get('ball_x');by=get('ball_y')
    outside=seen & ((bx<-0.01)|(bx>1.01)|(by<-0.01)|(by>1.01))
    z[:,2]+=p['outside_bonus']*outside
    return z / max(p['temperature'],1e-6)

def softmax(z):
    x=z-z.max(axis=1,keepdims=True);e=np.exp(x);return e/e.sum(axis=1,keepdims=True)

def majority_vote(labels,half_width=50,blocks=HALVES):
    result=np.full(len(labels),'X',dtype='<U1')
    for lo,hi in blocks:
        sample=np.asarray(labels[lo:hi+1]);n=len(sample)
        if half_width==0:result[lo:hi+1]=sample;continue
        c=np.cumsum(np.column_stack([sample=='H',sample=='A',sample=='D']),axis=0)
        c=np.vstack([np.zeros(3,dtype=int),c]);ts=np.arange(n)
        counts=c[np.minimum(n,ts+half_width+1)]-c[np.maximum(0,ts-half_width)]
        result[lo:hi+1]=CODES[np.argmax(counts,axis=1)]
    return result

def nearest_rule(df):
    h=df['home_nearest_ball_dist'].to_numpy();a=df['away_nearest_ball_dist'].to_numpy()
    valid=np.isfinite(h) & np.isfinite(a) & (df['ball_visible'].to_numpy()>0)
    out=np.full(len(df),'D',dtype='<U1');out[valid]=np.where(h[valid]<=a[valid],'H','A')
    return out

def viterbi(logits,blocks=HALVES,penalties=(13.0,10.0,7.0),dynamic=None):
    '''Viterbi on discriminative observation potentials; penalties for HA, HD, DH.
       Not a trained generative HMM. Softmax optional because row constant cancels.'''
    out=np.full(len(logits),'X',dtype='<U1')
    ha,hd,dh=penalties
    transition=np.array([[0,-ha,-hd],[-ha,0,-hd],[-dh,-dh,0]],dtype=float)
    for low,high in blocks:
        seq=np.asarray(logits[low:high+1]);n=len(seq)
        score=seq[0]+np.log([.45,.45,.10]);back=np.zeros((n,3),dtype=np.uint8)
        for t in range(1,n):
            tr=transition
            if dynamic is not None:
                # factor <1 softens switches at automatically detected transition cues.
                mult=float(dynamic[low+t]);tr=transition*mult
            cand=score[:,None]+tr
            back[t]=np.argmax(cand,axis=0)
            score=seq[t]+cand[back[t],np.arange(3)]
        states=np.empty(n,dtype=np.uint8);states[-1]=np.argmax(score)
        for t in range(n-1,0,-1):states[t-1]=back[t,states[t]]
        out[low:high+1]=CODES[states]
    return out

def enforce_published_blocks(labels):
    result=np.full(len(labels),'X',dtype='<U1')
    for lo,hi in HALVES: result[lo:hi+1]=np.asarray(labels)[lo:hi+1]
    result[HALVES[1][0]:SECOND_KICKOFF]='D'
    return result

def segments(labels,blocks=HALVES):
    out=[];a=np.asarray(labels)
    for lo,hi in blocks:
        v=a[lo:hi+1]
        if not len(v):continue
        cuts=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        out.extend((lo+int(cuts[i]),lo+int(cuts[i+1]),str(v[cuts[i]])) for i in range(len(cuts)-1))
    return out

def diagnostics(labels,blocks=HALVES):
    seg=[(s,e,k) for s,e,k in segments(labels,blocks) if k in ('H','A')]
    d=np.array([e-s for s,e,_ in seg])/FPS
    covered=np.concatenate([np.asarray(labels)[lo:hi+1] for lo,hi in blocks])
    return dict(n_possessions=len(seg),mean_possession_s=round(float(d.mean()),2) if len(d) else 0,
                possessions_lt_1s=int((d<1).sum()),D_pct=round(float((covered=='D').mean())*100,2),
                H_pct=round(float((covered=='H').mean())*100,2),A_pct=round(float((covered=='A').mean())*100,2))

def save_submission(labels,name,apply_blocks=True):
    a=enforce_published_blocks(labels) if apply_blocks else np.asarray(labels,dtype='<U1')
    if len(a)!=N_FRAMES or not np.isin(a,['H','A','D','X']).all(): raise ValueError('CSV non conforme')
    out=OUTPUT_DIR/name
    pd.DataFrame({'frame':np.arange(len(a),dtype=np.int32),'label':a}).to_csv(out,index=False)
    tab=pd.read_csv(out)
    assert tab.columns.tolist()==['frame','label'] and len(tab)==N_FRAMES
    assert np.array_equal(tab['frame'].to_numpy(),np.arange(N_FRAMES))
    print('Soumission validée (format local):',out)
    return out

def view_models(df,predictions,begin=11185,seconds=32):
    e=min(len(df),begin+int(seconds*FPS));t=np.arange(begin,e)/FPS
    fig,axs=plt.subplots(2,1,figsize=(14,6),sharex=True)
    axs[0].plot(t,df['nearest_dist_diff'].to_numpy()[begin:e],linewidth=0.9)
    axs[0].axhline(0,color='grey',ls='--');axs[0].set_ylabel('d_H − d_A (m)')
    k={'H':0,'A':1,'D':2,'X':3}
    for i,(name,p) in enumerate(predictions.items()):
        y=np.array([k[v] for v in p[begin:e]])+4*i
        axs[1].step(t,y,where='post',label=name)
    axs[1].set_yticks(np.arange(len(predictions))*4,[*predictions]);axs[1].set_xlabel('Temps vidéo (s)')
    axs[1].set_title('La comparaison des prédictions ne fournit PAS la vérité terrain')
    plt.tight_layout();plt.show()

In [ ]:
features_raw, raw = load_match()
assert len(features_raw)==199319 and (features_raw.frame.to_numpy()==np.arange(N_FRAMES)).all()
print('Taille des données:',features_raw.shape,'| ballon invisible:',f'{100*(features_raw.ball_visible==0).mean():.2f}%')
display(features_raw[['frame','ball_visible','nearest_dist_diff','closest_distance','tag_setpiece']].iloc[11185:11193])

## 2 · Modèle de référence : proximité, scores H/A/D et Viterbi

Les scores d'observation de la v1 sont **discriminatifs**, non des probabilités HMM apprises. La proximité oriente H/A, les tags offensifs ajoutent des indices et le biais `dead_bias=-0.7` freine `D`. La programmation dynamique choisit la séquence qui maximise la somme des scores moins les coûts de transition. `switch_strong` garde les mêmes features que `baseline_hmm` et augmente **uniquement** les coûts de transition `(13,10,7)→(23,17,11)`.

La température `T=3` sera testée **séparément** : diviser tous les logits par 3 revient à augmenter relativement le coût de transition. Ce n'est pas équivalent à une probabilité de persistance de 0,999 sans normaliser une vraie matrice HMM.

In [ ]:
MASK_GRADED=np.zeros(N_FRAMES,dtype=bool)
for lo,hi in HALVES:MASK_GRADED[lo:hi+1]=True
MASK_FLEX=MASK_GRADED.copy();MASK_FLEX[HALVES[1][0]:SECOND_KICKOFF]=False
nearest=enforce_published_blocks(nearest_rule(features_raw))
z_base0=make_logits(features_raw)
base_hmm=enforce_published_blocks(viterbi(z_base0,penalties=PEN_BASE))
switch_strong=enforce_published_blocks(viterbi(z_base0,penalties=PEN_STRONG))
display(pd.DataFrame([{'modele':k,**diagnostics(v)} for k,v in [('nearest',nearest),('baseline_hmm',base_hmm),('switch_strong',switch_strong)]]))

## 3 · Réparer les trous **courts** du ballon (sans inventer de contrôle)

Nous interpolons seulement un trou **entièrement encadré** et d'une durée totale ≤ 1 seconde. Nous n'interpolons pas une partie d'un trou long. Un point interpolé a une qualité faible et ne peut pas, à lui seul, créer un nouveau contrôle. Les trajectoires aériennes et les rebonds restent une source d'erreur. Les features sont recalculées pour les distances interpolées, sans confondre interpolation et observation originale.

In [ ]:
# Research-inspired implementations: explain these are proxies, not exact papers.

def fill_short_ball_gaps(df, players, teams, max_gap=20):
    '''Only bridge short two-sided occlusions, within each half; no long extrapolation.
       Inferred positions are hypotheses and never override original observations.'''
    n=len(df);x=df['ball_x'].to_numpy(float);y=df['ball_y'].to_numpy(float)
    valid=np.isfinite(x)&np.isfinite(y)
    inferred_x=np.full(n,np.nan);inferred_y=np.full(n,np.nan);quality=np.zeros(n)
    for lo,hi in HALVES:
        v=valid[lo:hi+1]
        changes=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for i in range(len(changes)-1):
            a,b=int(changes[i]),int(changes[i+1]);size=b-a
            if v[a] or size>max_gap or a==0 or b==len(v):continue
            left=lo+a-1;right=lo+b
            if not(valid[left] and valid[right]):continue
            for coord,buf in ((x,inferred_x),(y,inferred_y)):
                buf[lo+a:lo+b]=np.interp(np.arange(lo+a,lo+b),[left,right],[coord[left],coord[right]])
            quality[lo+a:lo+b]=max(.1,1-size/(max_gap+1))
    idx=np.flatnonzero(quality>0)
    # group-specific minimal distances to imputed ball positions
    for group,name in ((0,'home'),(1,'away')):
        p=players[idx][:,teams==group,:2]
        dx=(p[:,:,0]-inferred_x[idx,None])*LENGTH_M
        dy=(p[:,:,1]-inferred_y[idx,None])*WIDTH_M
        d=np.sqrt(dx*dx+dy*dy)
        d[~np.isfinite(d)]=np.inf
        mins=d.min(axis=1);mins[~np.isfinite(mins)]=np.nan
        full=np.full(n,np.nan);full[idx]=mins
        df[name+'_imputed_nearest']=full
    df['ball_gap_quality']=quality
    df['ball_x_inferred']=inferred_x;df['ball_y_inferred']=inferred_y
    return df

def graph_context_logits(df,base=None,pressure_weight=.55,imputation_weight=.75):
    '''Set/graph-inspired collective pressure around ball; no learned GNN claimed.'''
    z=make_logits(df) if base is None else np.array(base,copy=True)
    def arr(k):return pd.to_numeric(df[k],errors='coerce').to_numpy(float)
    # Relational features: number of nearby teammates and second-nearest distance.
    h=np.nan_to_num(arr('home_players_within_5m'),nan=0.0)
    a=np.nan_to_num(arr('away_players_within_5m'),nan=0.0)
    diff=np.clip(np.log1p(h)-np.log1p(a),-2,2)
    seen=arr('ball_visible')>.5
    z[:,0]+=pressure_weight*diff*seen
    z[:,1]-=pressure_weight*diff*seen
    h2=arr('home_second_nearest_ball_dist');a2=arr('away_second_nearest_ball_dist')
    ok=seen&np.isfinite(h2)&np.isfinite(a2)
    extra=np.zeros(len(df));extra[ok]=np.clip((a2[ok]-h2[ok])/8,-1,1)
    z[:,0]+=.32*extra;z[:,1]-=.32*extra
    if 'ball_gap_quality' in df:
        q=arr('ball_gap_quality')
        ih=arr('home_imputed_nearest');ia=arr('away_imputed_nearest')
        valid=(q>0)&np.isfinite(ih)&np.isfinite(ia)
        adv=np.zeros(len(df));adv[valid]=np.clip((ia[valid]-ih[valid])/5,-1.5,1.5)*q[valid]
        z[:,0]+=imputation_weight*adv;z[:,1]-=imputation_weight*adv
    return z

def multiscale_logits(logits,blocks=HALVES,scales=(3,15,50),weights=(.35,.40,.25)):
    '''Gaussian temporal aggregations, a TCN-inspired feature, not a learned TCN.'''
    x=np.asarray(logits,float)
    y=np.zeros_like(x)
    for lo,hi in blocks:
        a=x[lo:hi+1]
        out=np.zeros_like(a)
        for sigma,w in zip(scales,weights):
            out+=w*gaussian_filter1d(a,sigma=sigma,axis=0,mode='nearest')
        y[lo:hi+1]=out
    return y

def transition_cues(df):
    '''Automatic generic change cues, no frame-specific hand constants.'''
    n=len(df);factor=np.ones(n)
    diff=pd.to_numeric(df['nearest_dist_diff'],errors='coerce').to_numpy(float)
    finite=np.isfinite(diff);delta=np.zeros(n)
    ok=finite[1:] & finite[:-1]
    delta[1:]=np.where(ok,np.minimum(np.abs(np.diff(np.nan_to_num(diff,nan=0)))/5,1),0)
    # soften transition penalty around sharp changes, but only a little
    factor-=.12*delta
    tags=(df['tag_transition_home'].to_numpy()>0)|(df['tag_transition_away'].to_numpy()>0)
    factor[tags]*=.80
    factor=np.clip(factor,.65,1.15)
    return factor

def pseudo_train_classifier(df,graph_z,limit_per_class=12000):
    '''Entirely AUTOMATIC pseudo labels from the match, not GT: self-distillation experiment.
       Classifier can amplify baseline biases; does not imply genuine classification accuracy.'''
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier
    except ImportError:
        print('sklearn absent: pseudo-training désactivé');return None,None,None
    # Confidence of heuristic model; this is NOT a calibrated posterior.
    sm=multiscale_logits(graph_z,scales=(4,15,40),weights=(.30,.40,.30))
    probs=softmax(sm)
    labels=np.argmax(probs,axis=1)
    conf=np.max(probs,axis=1)
    # No part of manually annotated or public GT is consulted.
    cols=['nearest_dist_diff','home_nearest_ball_dist','away_nearest_ball_dist',
          'closest_distance','home_mean_speed','away_mean_speed','ball_speed',
          'tag_attack_home','tag_attack_away','tag_setpiece',
          'home_players_within_2m','away_players_within_2m',
          'home_players_within_5m','away_players_within_5m',
          'home_second_nearest_ball_dist','away_second_nearest_ball_dist','ball_gap_quality']
    cols=[c for c in cols if c in df]
    X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy(np.float32)
    for lo,hi in HALVES:
        pass  # Restriction appliquée dans mask ci-dessous
    inplay=np.zeros(len(df),bool)
    for lo,hi in HALVES:inplay[lo:hi+1]=True
    mask=inplay&(conf>=.68)
    indices=[];rng=np.random.default_rng(2027)
    # H/A are the main self-supervised target; D remains rule-based if too few safe pseudo examples.
    classes=[k for k in range(3) if int(np.sum(mask&(labels==k)))>=100]
    if not all(k in classes for k in (0,1)):
        print('Pseudo-labels H/A insuffisants: auto-apprentissage ignoré');return None,None,None
    if 2 not in classes:print('Pseudo-étiquette D trop rare: entraînement H/A seulement, D reste physique/heuristique.')
    for k in classes:
        idk=np.flatnonzero(mask&(labels==k))
        indices.extend(rng.choice(idk,size=min(limit_per_class,len(idk)),replace=False))
    ix=np.array(indices)
    model=HistGradientBoostingClassifier(max_iter=85,max_leaf_nodes=15,
       learning_rate=.08,l2_regularization=1.5,random_state=2027)
    model.fit(X[ix],labels[ix])
    p=np.full((len(X),3),1e-5,dtype=float)
    predicted=model.predict_proba(X)
    p[:,model.classes_.astype(int)]=predicted
    # With two-class self-training, keep a small neutral D mass; the fused model
    # below only consumes the H-vs-A log odds, not this artificial D probability.
    p/=p.sum(axis=1,keepdims=True)
    print('Pseudo-labels automatiques:',{CODES[k]:int((labels[ix]==k).sum()) for k in range(3)},
          '| PAS une vérité terrain')
    return model,p,cols

def inspect_video_sample(video_path,at_frames=(11185,11190,11215,15000)):
    '''Optional video sanity check, no hand-labelled outputs or model supervision.'''
    if not Path(video_path).exists():
        print('Vidéo absente sur Kaggle : branche vidéo ignorée, modèles tracking toujours exécutables.')
        return
    try:import cv2
    except ImportError:
        print('OpenCV absent : aperçu vidéo ignoré');return
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print('Vidéo non lisible : aperçu ignoré');return
    fig,axs=plt.subplots(1,len(at_frames),figsize=(15,3))
    for ax,i in zip(axs,at_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(i));ret,frame=cap.read()
        if ret:ax.imshow(cv2.cvtColor(cv2.resize(frame,(480,270)),cv2.COLOR_BGR2RGB))
        ax.set_title(f'Image vidéo {i}');ax.axis('off')
    cap.release();plt.tight_layout();plt.show()
    print('Inspection uniquement : ne pas produire d’annotations humaines du match.')

In [ ]:
features=fill_short_ball_gaps(features_raw,raw['players'],raw['team'],max_gap=25)
print('Frames interpolées :',int((features.ball_gap_quality>0).sum()))
display(features[['ball_visible','ball_gap_quality','home_imputed_nearest','away_imputed_nearest']].describe().round(2))

## 4 · Lois du jeu et détection des contrôles : implémentation contrôlée

### 4.1 Contrôle exclusif et mémoire du dernier contrôleur

Une passe ne change pas automatiquement la possession. Un candidat au contrôle H exige un joueur H à moins de **1,8 m**, aucun A dans ce rayon, et une configuration stable pendant **au moins 5 frames à 25 FPS** (durées remises à l'échelle pour SkillCorner 10 FPS). Les ballons rapides (>16 m/s) ne déclenchent pas un nouveau contrôle. En cas d'occlusion, l'identité du dernier contrôleur est propagée au maximum **10 secondes**, avec le poids `1.5·exp(-âge/6 secondes)`.

Les événements de remise en jeu mettent à zéro cette mémoire afin de ne pas transférer artificiellement l'ancien possesseur au nouveau coup franc. Ce module utilise uniquement les observations **réelles** du ballon pour déclencher un nouveau contrôle, jamais les points interpolés.

### 4.2 Duels, touches, reprises

- **Duel** : si un joueur de chaque équipe est à moins de 2 m, on atténue le signal brut de proximité au profit de la mémoire et de la continuité temporelle.
- **Touche de balle candidate** : changement de vitesse du ballon et proximité avec un joueur ; ce n'est pas un détecteur entraîné de contact, mais un indice complémentaire pré-déclaré.
- **Remises en jeu** : `Throw-in`, `Corner Kick`, `Goal Kick`, `Free Kick`, `Kick Off` déclenchent un reset. Le `Team` d'une remise n'est pas ajouté comme priorité tant que son sens n'est pas confirmé par l'accord avec les contrôles automatiques.
- **Défense** : le champ `Team` des événements `DEFENDING` est audité, **pas utilisé comme vérité** lorsque sa sémantique est ambiguë.

In [ ]:
RESTART_TYPES={'throw-in','corner kick','goal kick','free kick','kick off (after goal)','kick off (start)'}

def canon_type(s):
    return ' '.join(str(s or '').lower().replace('_',' ').replace('–','-').split())

def restart_marks(events,n):
    reset=np.zeros(n,bool); context=np.zeros(n,bool);priH=np.zeros(n,float);priA=np.zeros(n,float)
    types=[]
    for ev in events or []:
        typ=canon_type(ev.get('type',''))
        if ev.get('code','')!='SET PIECES' or typ not in RESTART_TYPES:continue
        a=max(0,min(n-1,int(ev['lo']))); b=max(a,min(n-1,int(ev['hi'])))
        reset[a]=True;reset[b]=True
        context[a:b+1]=True
        types.append((typ,ev.get('side','?'),a,b))
        # Team prior volontairement NON activé : à confirmer après diagnostic.
    return reset,context,priH,priA,types

def ball_dynamics(df,blocks,fps,field=(105.3,68.0)):
    n=len(df);x=pd.to_numeric(df['ball_x'],errors='coerce').to_numpy(float)
    y=pd.to_numeric(df['ball_y'],errors='coerce').to_numpy(float)
    seen=df.ball_visible.to_numpy(float)>0.5
    speed=np.full(n,np.nan); accel=np.full(n,np.nan)
    # Source initiale ATD conservée si disponible, sinon calcul à partir des points détectés.
    atd=(pd.to_numeric(df['ball_speed'],errors='coerce').to_numpy(float)
         if 'ball_speed' in df else np.full(n,np.nan))
    for lo,hi in blocks:
        if hi<=lo:continue
        i=np.arange(lo+1,hi+1)
        valid=seen[i]&seen[i-1]&np.isfinite(x[i])&np.isfinite(x[i-1])&np.isfinite(y[i])&np.isfinite(y[i-1])
        if 'frame' in df:
            f=df.frame.to_numpy();valid &= (f[i]==f[i-1]+1)
        v=np.hypot((x[i]-x[i-1])*field[0],(y[i]-y[i-1])*field[1])*fps
        speed[i[valid]]=v[valid]
        # conserver la vitesse ATD quand elle correspond à une observation réelle
        take=seen[lo:hi+1]&np.isfinite(atd[lo:hi+1])&(atd[lo:hi+1]>=0)
        window=speed[lo:hi+1];window[take]=atd[lo:hi+1][take]
        accel[i[1:]]=np.where(np.isfinite(speed[i[1:]])&np.isfinite(speed[i[:-1]]),
                               (speed[i[1:]]-speed[i[:-1]])*fps,np.nan)
    return speed,accel

def controller_evidence(df,blocks,fps,events=None,r=R_CTRL,min_len=CTRL_MIN_LEN,
                        max_age_sec=CTRL_MAX_AGE_SEC,ball_speed_max=CTRL_MAX_BALL_SPEED):
    n=len(df);dH=df.home_nearest_ball_dist.to_numpy(float);dA=df.away_nearest_ball_dist.to_numpy(float)
    seen=df.ball_visible.to_numpy(float)>0.5
    speed,accel=ball_dynamics(df,blocks,fps)
    reset,context,priorH,priorA,types=restart_marks(events,n)
    min_frames=max(2,int(np.ceil(min_len*fps/25.)))
    max_age=int(max_age_sec*fps)
    direct=np.full(n,-1,np.int8);age=np.full(n,np.inf);carried=np.full(n,-1,np.int8)
    witness=np.zeros(n,bool);touch=np.zeros(n,bool);duel=np.zeros(n,bool)
    h2=df.home_players_within_2m.to_numpy(float) if 'home_players_within_2m' in df else np.zeros(n)
    a2=df.away_players_within_2m.to_numpy(float) if 'away_players_within_2m' in df else np.zeros(n)
    duel=(h2>0)&(a2>0)&seen
    cH=seen&np.isfinite(dH)&(dH<=r)&(~np.isfinite(dA)|(dA>r))
    cA=seen&np.isfinite(dA)&(dA<=r)&(~np.isfinite(dH)|(dH>r))
    # La vitesse inconnue ne devient jamais arbitrairement 0 ; on utilise le signal connu seulement.
    fast=np.isfinite(speed)&(speed>ball_speed_max)
    provisional=np.full(n,-1,np.int8)
    provisional[cH&~fast]=0;provisional[cA&~fast]=1
    # Toucher candidat = rupture de vitesse ET proximité réelle ET contrôle exclusif.
    touch=(seen&~fast&np.isfinite(accel)&(np.abs(accel)>TOUCH_ACCEL)&
           (np.fmin(dH,dA)<r)&(provisional>=0))
    strength=np.zeros((n,2),float);status=np.zeros(n,np.int8)
    for lo,hi in blocks:
        # reset block prevents cross-half propagation
        last=-1;last_frame=-10**9
        v=provisional[lo:hi+1];cut=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for s,e in zip(cut[:-1],cut[1:]):
            if v[s]>=0 and (e-s)>=min_frames and not np.any(reset[lo+s:lo+e]):
                direct[lo+s:lo+e]=v[s]
        for t in range(lo,hi+1):
            if reset[t]:last=-1;last_frame=-10**9
            if direct[t]>=0:
                last=int(direct[t]);last_frame=t;witness[t]=True
            elapsed=t-last_frame
            if last>=0 and elapsed<=max_age:
                carried[t]=last;age[t]=elapsed/fps
                w=1.5*np.exp(-age[t]/6.)
                strength[t,last]=w
        status[lo:hi+1]=(carried[lo:hi+1]>=0).astype(np.int8)
    detail=pd.DataFrame({'ctrl_direct':direct,'last_controller':carried,'controller_age_s':age,
             'touch_candidate':touch,'duel':duel,'restart_reset':reset,'restart_context':context,
             'ball_speed_mps':speed,'ball_accel_mps2':accel,'controller_observed':witness})
    return strength,detail,types

def controller_variants(df,blocks,fps,events=None,raw_players=None,team_codes=None):
    z=make_logits(df)
    # Séparation stricte de l'ablation : les modèles sans « restart » n'utilisent PAS les tags.
    strength,detail0,_=controller_evidence(df,blocks,fps,events=None)
    strength_restart,detail,types=(controller_evidence(df,blocks,fps,events=events)
                                   if events else (strength,detail0,[]))
    # modèle 1 : mémoire du contrôleur, aucun changement de pénalité temporelle ni reset
    z_control=z.copy();z_control[:,:2]+=strength
    # modèle 2 : atténuer proximité en duel, sans enlever la mémoire
    z_duel=z_control.copy();duel=detail0.duel.to_numpy(bool)
    # Retirer 50 % de la composante de PROXIMITÉ seulement ; conserver les tags offensifs.
    rawdiff=pd.to_numeric(df['nearest_dist_diff'],errors='coerce').to_numpy(float)
    prox=np.zeros(len(df),float)
    valid=np.isfinite(rawdiff)&(df.ball_visible.to_numpy(float)>.5)
    prox[valid]=np.clip(rawdiff[valid]/DEFAULT_PARAMS['prox_scale'],
                        -DEFAULT_PARAMS['prox_clip'],DEFAULT_PARAMS['prox_clip'])
    z_duel[duel,0]+=.5*prox[duel]
    z_duel[duel,1]-=.5*prox[duel]
    # modèle 3 : candidate touch adds H/A evidence inside controller strength already;
    # for pure ablation separate it here via a small extra boost on direct touch frames.
    z_touch=z_duel.copy();hit=detail0.touch_candidate.to_numpy(bool)
    direct=detail0.ctrl_direct.to_numpy()
    z_touch[hit&(direct==0),0]+=.4
    z_touch[hit&(direct==1),1]+=.4
    # modèle 4 : reset exists in controller, consider restarts as a separate D candidate
    z_restart=z_touch.copy();reset=detail.restart_reset.to_numpy(bool)
    # retirer l'ancienne mémoire, puis ajouter la mémoire explicitement réinitialisée
    z_restart[:,:2]+=strength_restart-strength
    if events is not None:
        z_restart[reset,2]+=.4
    # Control+multiscale: apply group feature evidence, but do not double count strength
    graph=graph_context_logits(df,z,pressure_weight=.55,imputation_weight=.75)
    factor=fps/25.
    multi=multiscale_logits(graph,blocks=blocks,scales=tuple(max(.5,s*factor) for s in (3,15,50)),weights=(.35,.40,.25))
    fused=.65*z_touch+.35*multi
    fused[:,:2]+=.35*strength
    return dict(base=z,control=z_control,duel=z_duel,touch=z_touch,restart=z_restart,
                graph=graph,multi=multi,fused=fused,detail=detail,
                detail_no_restart=detail0,types=types)

### 4.3 Fractions de joueurs immobiles et contrôles contradictoires

La fraction de joueurs à moins de **0,7 m/s** sur une fenêtre de **2 secondes** peut accompagner un arrêt de jeu. Elle est calculable avec les vitesses individuelles ATD de Metrica. Sur SkillCorner, notre lecteur externe actuel ne retient que les vitesses moyennes d'équipe : cette variable n'est **pas équivalente** ; elle ne doit donc pas intervenir dans une sélection cross-provider tant qu'on n'a pas implémenté le même calcul de part et d'autre.

Nous utilisons également le contrôle observé comme témoin automatique pour auditer le champ `Team` des événements `DEFENDING` ; tant que son sens est ambigu, aucune règle offensive/défensive ne dépend de ce champ.

In [ ]:
def stationary_player_fraction(players,team_codes,fps=25.,threshold=.7,window_s=2.):
    v=players[:,:,2].astype(float)
    valid=np.isfinite(v)&(v>=0)
    stationary=np.divide(np.sum(valid&(v<threshold),axis=1),np.sum(valid,axis=1),
                  out=np.full(len(v),np.nan),where=np.sum(valid,axis=1)>0)
    roll=pd.Series(stationary).rolling(max(2,round(window_s*fps)),min_periods=round(.5*window_s*fps)).mean()
    return roll.to_numpy(float)

STATIONARY_FRACTION=stationary_player_fraction(raw['players'],raw['team'],FPS)
print('Fraction de joueurs immobiles, calcul ATD uniquement :')
display(pd.Series(STATIONARY_FRACTION[MASK_FLEX]).describe().round(3))

def event_team_agreement(events,detail,codes,lookahead=50):
    direct=detail.ctrl_direct.to_numpy(int);rows=[]
    for ev in events:
        if ev.get('side') not in ('H','A'):continue
        if ev.get('code') not in codes:continue
        a=max(0,min(len(direct)-1,int(ev['hi'])+1));b=min(len(direct),a+lookahead)
        known=direct[a:b];known=known[known>=0]
        if not len(known):continue
        side='H' if known[0]==0 else 'A'
        rows.append({'code':ev['code'],'type':ev.get('type',''),
                     'team_tag':ev['side'],'first_auto_control':side,'agreement':ev['side']==side})
    return pd.DataFrame(rows)

base_evid=controller_variants(features,HALVES,FPS,events=raw['events'])
print('Reset des remises en jeu :',int(base_evid['detail'].restart_reset.sum()),'| types :')
display(pd.Series([v[0] for v in base_evid['types']]).value_counts().rename_axis('type').to_frame('evenements'))
print('Audit Team pour DEFENDING — indicatif, aucune règle automatique activée :')
_agreement=event_team_agreement(raw['events'],base_evid['detail'],
             {ev.get('code') for ev in raw['events'] if ev.get('code','').startswith('DEFENDING ')})
if len(_agreement): display(_agreement.groupby('code').agg(n=('agreement','size'),accord=('agreement','mean')).round(3))
else:print('Pas assez de contrôles observés pour cet audit.')
print('Contrôles observés',base_evid['detail'].controller_observed.sum(),
      '| phases duels',base_evid['detail'].duel.sum(),'| touches candidates',base_evid['detail'].touch_candidate.sum())

## 5 · Détecteur de jeu arrêté D V2, distinct des modèles H/A

La classe `D` correspond au ballon **hors jeu**, y compris une faute sifflée, un corner à exécuter ou une touche, **pas** à la règle du hors-jeu d'un attaquant. Un joueur proche d'un ballon immobile est précisément compatible avec un coup franc. Un ballon invisible sans autre preuve ne fait pas monter le score D.

Le score V2 utilise **sortie confirmée**, **préparation de reprise**, **immobilité près des lignes**, **mémoire d'arrêt**, pénalité pour **jeu actif**, **reprise détectée** et **joueur proche d'un ballon en mouvement**. Les indices sont heuristiques ; le vrai rappel `D` ne peut pas être appris ou validé avec les seuls labels H/A de SkillCorner. Deux familles restent séparées : `switch_strong` (D ancien) et `D_V2_strong` (H/A identiques, D remplacé), pour isoler l'effet D.

In [ ]:
V2_PARAMS=dict(bias=-.70,boundary_m=2.,ball_still_mps=1.,ball_active_mps=4.,
               player_near_m=3.,quiet_player_mps=1.5,exit_margin_m=.15,memory_sec=8.)
def xc5_d_evidence(df, blocks, fps, params=V2_PARAMS, field=(105.3, 68.0)):
    """Evidence D heuristique, pas probabilite calibree ni classifieur entraine.
    Aucun calcul ne consulte de labels GT. Tous les calculs temporels sont locaux aux blocs.
    """
    n = len(df); P = params
    def x(c, default=np.nan):
        if c not in df: return np.full(n, default, float)
        return pd.to_numeric(df[c], errors='coerce').to_numpy(float)
    bx, by = x('ball_x'), x('ball_y')
    seen = (x('ball_visible', 0)>0.5) & np.isfinite(bx) & np.isfinite(by)
    near = x('closest_distance')
    sh, sa = x('home_mean_speed'), x('away_mean_speed')
    quiet = (np.nan_to_num(sh,nan=100)<P['quiet_player_mps']) & (np.nan_to_num(sa,nan=100)<P['quiet_player_mps'])
    tag = np.nan_to_num(x('tag_setpiece',0),nan=0)>0
    L,W = field
    dline = np.minimum.reduce([bx*L,(1-bx)*L,by*W,(1-by)*W])
    close_line = seen & (dline>=-0.5) & (dline<P['boundary_m'])
    outside_raw = seen & (dline < -P['exit_margin_m'])
    # Vitesse DERIVEE des positions consecutives; pas de derivee sur trou/periode.
    speed = np.full(n,np.nan)
    for lo, hi in blocks:
        if hi<=lo: continue
        ix=np.arange(lo+1,hi+1)
        consecutive = seen[ix] & seen[ix-1]
        if 'frame' in df:
            f=x('frame');consecutive &= f[ix]==f[ix-1]+1
        distance=np.hypot((bx[ix]-bx[ix-1])*L,(by[ix]-by[ix-1])*W)
        speed[ix[consecutive]]=distance[consecutive]*fps
    # Ne pas reutiliser une vitesse inexistante (ballon invisible).
    stationary=np.zeros(n,bool);active=np.zeros(n,bool)
    exit_confirmed=np.zeros(n,bool)
    for lo,hi in blocks:
        v=pd.Series(speed[lo:hi+1])
        smooth=v.rolling(max(2,round(.32*fps)),min_periods=max(2,round(.16*fps))).median().to_numpy()
        stationary[lo:hi+1]=seen[lo:hi+1] & np.isfinite(smooth) & (smooth<P['ball_still_mps'])
        active[lo:hi+1]=seen[lo:hi+1] & np.isfinite(smooth) & (smooth>P['ball_active_mps'])
        xx=outside_raw[lo:hi+1]
        exit_confirmed[lo:hi+1]=xx & (pd.Series(xx.astype(int)).rolling(3,min_periods=2).sum().to_numpy()>=2)
    close_player=seen & np.isfinite(near) & (near<P['player_near_m'])
    # 1) Setpiece doit etre combine a des indices physiques (tag seul trop large).
    prep=tag & stationary & (quiet | close_player)
    # 2) Au bord du terrain, ballon immobile + replacement -> suspicion moderee.
    line_prep=close_line & stationary & quiet
    # 3) En jeu: ballon se deplace, joueur en interaction, et pas hors des limites.
    alive=active & close_player & ~outside_raw
    # 4) Memoire courte APRES signal fort, effacee sur reprise plausible.
    memory=np.zeros(n,bool); restart=np.zeros(n,bool)
    ttl=max(1,int(P['memory_sec']*fps))
    for lo,hi in blocks:
        budget=0;move_count=0
        for t in range(lo,hi+1):
            if exit_confirmed[t] or prep[t]: budget=ttl
            restart_now=bool(alive[t] and (not tag[t] or budget>0))
            move_count=move_count+1 if restart_now else 0
            if budget>0 and move_count>=max(2,int(.20*fps)):
                restart[t]=True;budget=0
            memory[t]=budget>0
            budget=max(0,budget-1)
    # Pas de bonus D pour une simple disparition du ballon.
    # Eviter de penaliser un joueur proche d'un coup franc immobile.
    contributions = pd.DataFrame({
        'bias':np.full(n,P['bias']),
        'sortie_confirmee':2.2*exit_confirmed.astype(float),
        'preparation_reprise':1.5*prep.astype(float),
        'arret_pres_ligne':0.6*line_prep.astype(float),
        'memoire_arret':1.0*memory.astype(float),
        'jeu_actif':-1.4*alive.astype(float),
        'reprise_detectee':-1.0*restart.astype(float),
        'joueur_proche_en_mouvement':-0.45*(active & close_player).astype(float)
    })
    flags = pd.DataFrame({'ball_seen':seen,'ball_moving':active,'ball_still':stationary,
                          'near_line':close_line,'outside_confirmed':exit_confirmed,
                          'setpiece':tag,'prep_restart':prep,'quiet_players':quiet,
                          'alive':alive,'memory_stop':memory,'restart':restart,
                          'ball_speed_derived_mps':speed,'distance_to_line_m':dline})
    return contributions.sum(axis=1).to_numpy(), contributions, flags

def xc5_replace_d(z, dscore):
    out=np.asarray(z,float).copy();out[:,2]=dscore;return out

def xc5_D_segments(pred, blocks, fps):
    out=[]
    for lo,hi in blocks:
        if hi<lo:continue
        v=np.asarray(pred)[lo:hi+1]
        cuts=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for a,b in zip(cuts[:-1],cuts[1:]):
            if v[a]=='D':out.append((lo+int(a),lo+int(b)-1,(b-a)/fps))
    return out

# ========================================
# 1. TESTS UNITAIRES PHYSIQUES, SANS GT
# ========================================

In [ ]:
D_V2_SCORE,D_PARTS,D_FLAGS=xc5_d_evidence(features,HALVES,FPS)
print('Contributions du détecteur D V2 :')
display(D_PARTS[MASK_FLEX].agg(['mean','std']).T.round(3))
print('Indices géométriques :')
display((100*D_FLAGS[MASK_FLEX].select_dtypes(include='bool').mean()).round(2).to_frame('pct'))

## 6 · Modèles, mécanismes et comparaisons **sur les mêmes frames**

Nous instancions chaque variante avec le même décodeur lorsqu'on veut isoler la feature. Les variantes utilisant le score D V2 sont **hors compétition de sélection HA fondée sur SkillCorner**. Le modèle `control_restart_strong` bénéficie de l'existence des tags de Metrica et ne peut donc être départagé par SkillCorner où ils sont absents : il reste une ablation descriptive, pas un gagnant validé.

Nous appliquons un post-traitement *optionnel* : fusionner uniquement un segment H/A très court **entre deux segments du même label** ; un vrai transfert de possession court doit rester possible, donc cette règle doit être évaluée séparément.

In [ ]:
def merge_brief_islands(labels,blocks,fps,max_seconds=.5):
    output=np.asarray(labels,dtype='<U1').copy();threshold=max(1,round(max_seconds*fps))
    for lo,hi in blocks:
        v=output[lo:hi+1];cuts=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for k in range(1,len(cuts)-2):
            a,b=cuts[k],cuts[k+1]
            if b-a>threshold:continue
            left=v[cuts[k-1]];cur=v[a];right=v[cuts[k+1]]
            if cur in ('H','A') and left==right and left in ('H','A') and cur!=left:
                v[a:b]=left
    return output

def model_specs(obs,dscore=None,allow_restart=True):
    if dscore is None: dscore=np.full(len(obs['base']),V2_PARAMS['bias'])
    specs={
      'baseline_hmm':(obs['base'],PEN_BASE,None,False),
      'switch_strong':(obs['base'],PEN_STRONG,None,False),
      'temperature3_strong':(obs['base']/3.,PEN_TEMPERATURE_3,None,False),
      'control_strong':(obs['control'],PEN_STRONG,None,False),
      'control_duel_strong':(obs['duel'],PEN_STRONG,None,False),
      'control_touch_strong':(obs['touch'],PEN_STRONG,None,False),
      'graph_geometry':(obs['graph'],PEN_FRONTIER,None,False),
      'adaptive_multiscale':(obs['multi'],PEN_FRONTIER,None,False),
      'fusion_control_multiscale':(obs['fused'],PEN_STRONG,None,False),
      'fusion_control_postprocess':(obs['fused'],PEN_STRONG,None,True),
      'D_V2_strong':(xc5_replace_d(obs['base'],dscore),PEN_STRONG,None,False),
      'fusion_D_V2_strong':(xc5_replace_d(obs['fused'],dscore),PEN_STRONG,None,False)
    }
    if allow_restart:specs['control_restart_strong']=(obs['restart'],PEN_STRONG,None,False)
    return specs

def predict_specs(specs,blocks,fps,target=False):
    results={}
    for name,(z,pen,dyn,postprocess) in specs.items():
        if target:
            p=viterbi(z,blocks=blocks,penalties=pen,dynamic=dyn)
            p=enforce_published_blocks(p)
        else:
            p=xc03_decode(z,blocks,pen,dyn)
        if postprocess:
            p=merge_brief_islands(p,blocks,fps)
            if target:p=enforce_published_blocks(p)
        results[name]=p
    return results

TARGET_SPECS=model_specs(base_evid,D_V2_SCORE)
TARGET_PREDS=predict_specs(TARGET_SPECS,HALVES,FPS,target=True)
TARGET_PREDS['nearest']=nearest
TARGET_DIAG=pd.DataFrame([{'modele':name,**diagnostics(p)} for name,p in TARGET_PREDS.items()])
print('Diagnostiques descriptifs BAR-ATH (sans GT) :')
display(TARGET_DIAG.round(3))
TARGET_DIAG.to_csv(OUT/'XC_final_diagnostics_target_sans_GT.csv',index=False)

### 6.1 Matrice de désaccords, frames et transitions

Nous ne décidons pas « qui est vrai » sur la cible en regardant ses séquences. Le tableau mesure exclusivement l'effet des hypothèses (duel, contact, poids temporel, D V2). Les plus longues divergences signalent les faiblesses possibles de persistance et de calibration, sans constituer des labels.

In [ ]:
NAMES=list(TARGET_PREDS)
PAIR_DISAGREE=pd.DataFrame(index=NAMES,columns=NAMES,dtype=float)
for a in NAMES:
    for b in NAMES:
        PAIR_DISAGREE.loc[a,b]=100*np.mean(TARGET_PREDS[a][MASK_FLEX]!=TARGET_PREDS[b][MASK_FLEX])
display(PAIR_DISAGREE.round(2))
fig,ax=plt.subplots(figsize=(12,10));im=ax.imshow(PAIR_DISAGREE.values,cmap='magma',vmin=0,vmax=100)
ax.set_xticks(range(len(NAMES)),NAMES,rotation=60,ha='right');ax.set_yticks(range(len(NAMES)),NAMES)
fig.colorbar(im,ax=ax,label='Désaccord (%)');ax.set_title('Comparaison paire à paire — mêmes frames BAR-ATH, sans GT');plt.tight_layout();plt.show()

REF=TARGET_PREDS['switch_strong'];intervals=[]
for name,p in TARGET_PREDS.items():
    if name=='switch_strong':continue
    bad=MASK_FLEX&(p!=REF)
    cuts=np.r_[0,np.flatnonzero((bad[1:]!=bad[:-1]) | (p[1:]!=p[:-1]) | (REF[1:]!=REF[:-1]))+1,len(p)]
    for a,b in zip(cuts[:-1],cuts[1:]):
        if not bad[a] or b-a<5:continue
        intervals.append({'modele':name,'frame_debut':int(a),'frame_fin':int(b-1),
                          'timecode_s':round(a/FPS,2),'duree_s':round((b-a)/FPS,2),
                          'switch_strong':REF[a],'autre':p[a]})
DISAGREEMENTS=pd.DataFrame(intervals).sort_values('duree_s',ascending=False) if intervals else pd.DataFrame()
print('Plus longues divergences :')
display(DISAGREEMENTS.head(30))
PAIR_DISAGREE.to_csv(OUT/'XC_final_matrix_disagreements.csv')
DISAGREEMENTS.to_csv(OUT/'XC_final_intervalles_divergences.csv',index=False)

## 7 · Dataset externe SkillCorner et protocole de comparaison

Le tracking SkillCorner n'est **pas** celui des organisateurs et les définitions d'annotation diffèrent. Nous ne cherchons **pas** à reproduire le score officiel sur SkillCorner. Le proxy mesure uniquement les possessions **H/A connues** ; `D_on_known_HA_pct` mesure des faux `D` lorsque la possession H/A est annotée. Une frame sans H/A **n'est jamais automatiquement un vrai D**.

Nous comparons les variantes communes sur **cinq matchs complets**. Pour que le test soit équitable, toutes les méthodes utilisent les mêmes blocs temporels. Les features `Team` des Smart Tags sont absentes et ne sont jamais injectées clandestinement. L'inférence des nouveaux contrôles se base donc sur les données géométriques communes. La sélection se fait **uniquement** via les scores externes ; toute différence liée aux Smart Tags propres à Metrica est déclarée non validée par transfert.

In [ ]:
XC03_FPS=10.;XC03_REFERENCE_FPS=25.
XC03_OUT=OUT
XC03_CACHE=OUT/'xc_integral_skillcorner_cache';XC03_CACHE.mkdir(parents=True,exist_ok=True)
def xc03_locate(mid):
    name=f'{mid}_tracking_extrapolated.jsonl'
    roots=[Path('/kaggle/input'),Path('/kaggle/working'),XC03_CACHE]
    for root in roots:
        if not root.exists():continue
        for p in root.rglob(name):
            q=p.with_name(f'{mid}_match.json')
            if q.is_file() and p.stat().st_size>10000:
                with open(p,'rb') as handle:
                    if not handle.read(100).startswith(b'version https://git-lfs'):
                        return p,q
    return None,None

def xc03_fetch(url,dest):
    tmp=dest.with_suffix(dest.suffix+'.part')
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req,timeout=100) as source,open(tmp,'wb') as output:
            shutil.copyfileobj(source,output)
        if dest.name.endswith('tracking_extrapolated.jsonl'):
            if tmp.stat().st_size<10000 or tmp.open('rb').read(100).startswith(b'version https://git-lfs'):
                raise ValueError('Git LFS POINTER : pas les donnees reelles')
        tmp.replace(dest)
        return True
    except Exception as error:
        print('  Source indisponible :',str(error)[:150])
        tmp.unlink(missing_ok=True)
        return False

def xc03_source(mid):
    p,q=xc03_locate(mid)
    if p is not None:return p,q
    folder=XC03_CACHE/str(mid);folder.mkdir(exist_ok=True)
    p=folder/f'{mid}_tracking_extrapolated.jsonl'
    q=folder/f'{mid}_match.json'
    rel=f'data/matches/{mid}'
    if not q.is_file():
        if not xc03_fetch(f'https://raw.githubusercontent.com/SkillCorner/opendata/master/{rel}/{q.name}',q):
            raise RuntimeError(f'{mid} : métadonnées introuvables ; ajouter SkillCorner sur Kaggle')
    if not p.is_file() or p.stat().st_size<10000:
        urls=[f'https://media.githubusercontent.com/media/SkillCorner/opendata/refs/heads/master/{rel}/{p.name}',
              f'https://github.com/SkillCorner/opendata/raw/refs/heads/master/{rel}/{p.name}']
        if not any(xc03_fetch(url,p) for url in urls):
            raise RuntimeError(f'{mid} : tracking LFS indisponible ; Add Input SkillCorner')
    return p,q

# ================== FEATURES GEOMETRIQUES EXTERNES ==================
def xc03_float(x):
    try:return float(x) if x is not None else np.nan
    except (TypeError,ValueError):return np.nan

def xc03_side(x):
    s=str(x).lower().replace('_',' ').replace('-',' ').strip()
    return 'H' if s in ('home','home team','home goalkeeper') else ('A' if s in ('away','away team','away goalkeeper') else 'X')

def xc03_blocks(period,frame):
    cuts=np.r_[0,np.flatnonzero((period[1:]!=period[:-1]) | (frame[1:]!=frame[:-1]+1))+1,len(frame)]
    return [(int(cuts[i]),int(cuts[i+1]-1)) for i in range(len(cuts)-1)]

def xc03_load(tracking,metadata):
    with open(metadata,encoding='utf8') as handle:meta=json.load(handle)
    home_id=int(meta['home_team']['id']);away_id=int(meta['away_team']['id'])
    mapping={}
    for pl in meta.get('players',[]):
        team=0 if pl.get('team_id')==home_id else 1 if pl.get('team_id')==away_id else None
        if team is not None:
            for k in ('id','player_id','trackable_object'):
                if pl.get(k) is not None:mapping[pl[k]]=team
    length=xc03_float(meta.get('pitch_length',105.0));width=xc03_float(meta.get('pitch_width',68.0))
    if not np.isfinite(length):length=105.0
    if not np.isfinite(width):width=68.0
    rows=[];labels=[];periods=[];frames=[];positions=[]
    prev_positions={};prev_frame=None;prev_period=None;player_mapped=0
    with open(tracking,encoding='utf8') as handle:
        for line in handle:
            if not line.strip():continue
            rec=json.loads(line)
            f=int(rec['frame']);per=int(rec.get('period') or 1)
            possession=rec.get('possession') or {}
            labels.append(xc03_side(possession.get('group') if isinstance(possession,dict) else None))
            frames.append(f);periods.append(per)
            bd=rec.get('ball_data') or {}
            bx=xc03_float(bd.get('x'));by=xc03_float(bd.get('y'))
            detected=bd.get('is_detected',bd.get('is_detected_ball'))
            seen=bool(detected) if detected is not None else bool(np.isfinite(bx) and np.isfinite(by))
            seen=bool(seen and np.isfinite(bx) and np.isfinite(by))
            if not seen:bx=by=np.nan
            group=[[],[]];speed=[[],[]];now={}
            successive=(prev_period==per and prev_frame is not None and f-prev_frame==1)
            for player in rec.get('player_data') or []:
                pid=player.get('player_id',player.get('trackable_object'))
                t=mapping.get(pid)
                if t is None and player.get('team_id') is not None:
                    t=0 if player['team_id']==home_id else 1 if player['team_id']==away_id else None
                if t is None:continue
                x=xc03_float(player.get('x'));y=xc03_float(player.get('y'))
                if not(np.isfinite(x) and np.isfinite(y)):continue
                player_mapped+=1
                group[t].append((x,y))
                if pid is not None:
                    now[pid]=(x,y)
                    if successive and pid in prev_positions:
                        ox,oy=prev_positions[pid]
                        v=np.hypot(x-ox,y-oy)*XC03_FPS
                        if v<15:speed[t].append(v)
            prev_positions=now;prev_frame=f;prev_period=per
            dist=[];dist2=[];within2=[];within5=[];mean_speed=[]
            for t in (0,1):
                coord=np.asarray(group[t],float).reshape(-1,2)
                d=np.sort(np.hypot(coord[:,0]-bx,coord[:,1]-by)) if seen and len(coord) else np.array([])
                dist.append(float(d[0]) if len(d) else np.nan)
                dist2.append(float(d[1]) if len(d)>1 else np.nan)
                within2.append(float(np.sum(d<2)) if seen and len(d) else np.nan)
                within5.append(float(np.sum(d<5)) if seen and len(d) else np.nan)
                mean_speed.append(float(np.mean(speed[t])) if speed[t] else np.nan)
            positions.append(group)
            rows.append(dict(frame=f,ball_visible=int(seen),ball_x=bx/length+.5 if seen else np.nan,
                ball_y=by/width+.5 if seen else np.nan,ball_speed=np.nan,
                home_nearest_ball_dist=dist[0],away_nearest_ball_dist=dist[1],
                nearest_dist_diff=dist[0]-dist[1],closest_distance=np.fmin(dist[0],dist[1]),
                home_second_nearest_ball_dist=dist2[0],away_second_nearest_ball_dist=dist2[1],
                home_mean_speed=mean_speed[0],away_mean_speed=mean_speed[1],
                home_players_within_2m=within2[0],away_players_within_2m=within2[1],
                home_players_within_5m=within5[0],away_players_within_5m=within5[1]))
    df=pd.DataFrame(rows)
    order=np.argsort(np.asarray(frames),kind='stable')
    df=df.iloc[order].reset_index(drop=True)
    y=np.asarray(labels,dtype='<U1')[order]
    period=np.asarray(periods,dtype=int)[order]
    frame=np.asarray(frames,dtype=int)[order]
    positions=[positions[int(j)] for j in order]
    if np.mean(np.isin(y,['H','A']))<.08 or player_mapped<100:
        raise ValueError('Labels H/A ou associations des joueurs insuffisants : verifier le schema')
    for col in ('tag_attack_home','tag_attack_away','tag_setpiece','tag_transition_home',
                'tag_transition_away','tag_defensive_home','tag_defensive_away'):
        df[col]=0  # JAMAIS de GT mise en feature ; Smart Tags Metrica absents
    blocks=xc03_blocks(period,frame)
    # Equivalent physique du short-gap ball interpolation (20 frames a 25 FPS = 8 a 10 FPS).
    seen=df.ball_visible.to_numpy(bool)
    px=df.ball_x.to_numpy(float);py=df.ball_y.to_numpy(float)
    quality=np.zeros(len(df));ih=np.full(len(df),np.nan);ia=ih.copy()
    for left,right in blocks:
        v=seen[left:right+1]
        cuts=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for a,b in zip(cuts[:-1],cuts[1:]):
            size=b-a
            if v[a] or size>8 or a==0 or b==len(v):continue
            L=left+a-1;R=left+b
            if not(seen[L] and seen[R]):continue
            for index in range(left+a,left+b):
                alpha=(index-L)/(R-L)
                x=((1-alpha)*px[L]+alpha*px[R]-.5)*length
                yy=((1-alpha)*py[L]+alpha*py[R]-.5)*width
                quality[index]=max(.1,1-size/9)
                for t,buffer in ((0,ih),(1,ia)):
                    arr=np.asarray(positions[index][t],dtype=float).reshape(-1,2)
                    if len(arr):buffer[index]=np.min(np.hypot(arr[:,0]-x,arr[:,1]-yy))
    df['ball_gap_quality']=quality
    df['home_imputed_nearest']=ih
    df['away_imputed_nearest']=ia
    return dict(features=df,y=y,period=period,frame=frame,blocks=blocks,
        info=dict(id=int(meta['id']),home=meta['home_team']['name'],away=meta['away_team']['name'],
                  coverage_HA=float(np.mean(np.isin(y,['H','A']))),filled_gaps=int((quality>0).sum())))

# ================== DECODAGE VITERBI RAPIDE ==================
try:
    from numba import njit
    @njit
    def xc03_viterbi_core(z,mults,ha,hd,dh):
        n=len(z)
        costs=np.array([[0.,-ha,-hd],[-ha,0.,-hd],[-dh,-dh,0.]])
        back=np.zeros((n,3),dtype=np.uint8)
        score=np.array([z[0,0]+np.log(.45),z[0,1]+np.log(.45),z[0,2]+np.log(.10)])
        for t in range(1,n):
            nxt=np.empty(3)
            for k in range(3):
                best=-1e100;bestfrom=0
                for j in range(3):
                    value=score[j]+costs[j,k]*mults[t]
                    if value>best:best=value;bestfrom=j
                back[t,k]=bestfrom;nxt[k]=best+z[t,k]
            score=nxt
        states=np.empty(n,dtype=np.uint8)
        states[-1]=np.argmax(score)
        for t in range(n-1,0,-1):states[t-1]=back[t,states[t]]
        return states
    XC03_NUMBA=True
except ImportError:
    XC03_NUMBA=False


def xc03_decode(z,blocks,penalties,dynamic=None):
    # La même pénalité par unité de temps qu'en 25 FPS : observations * 25/10.
    obs=np.asarray(z,np.float64)*(XC03_REFERENCE_FPS/XC03_FPS)
    out=np.full(len(obs),'X',dtype='<U1')
    mults=np.ones(len(obs)) if dynamic is None else np.asarray(dynamic,np.float64)
    for lo,hi in blocks:
        if hi<lo:continue
        if XC03_NUMBA:
            state=xc03_viterbi_core(obs[lo:hi+1],mults[lo:hi+1],*map(float,penalties))
            out[lo:hi+1]=np.asarray(['H','A','D'])[state]
        else:
            # viterbi est deja fourni par le Notebook 03.
            out[lo:hi+1]=viterbi(obs,blocks=[(lo,hi)],penalties=penalties,
                dynamic=mults if dynamic is not None else None)[lo:hi+1]
    return out

# ================== DISTILLATION SANS LABELS ==================
def xc03_pseudo_boost(df,z_graph,blocks,fps=None):
    n=len(df);boost=np.zeros_like(z_graph)
    try:from sklearn.ensemble import HistGradientBoostingClassifier
    except ImportError:return boost,'sklearn indisponible'
    sigma=(XC03_FPS if fps is None else fps)/XC03_REFERENCE_FPS
    smooth=multiscale_logits(z_graph,blocks=blocks,scales=tuple(sigma*x for x in (4,15,40)),
                             weights=(.30,.40,.30))
    # Masquage aux vraies sequences temporelles, et pas aux labels externes.
    confidence=np.exp(smooth-smooth.max(axis=1,keepdims=True))
    confidence/=confidence.sum(axis=1,keepdims=True)
    pseudo=np.argmax(confidence,axis=1)
    conf=np.max(confidence,axis=1)
    ix=[];mask=conf>=.68
    rng=np.random.default_rng(2027)
    for k in (0,1,2):
        ids=np.flatnonzero(mask&(pseudo==k))
        if len(ids)>=100:ix.extend(rng.choice(ids,size=min(len(ids),12000),replace=False))
    ix=np.asarray(ix,dtype=int)
    if len(ix)==0 or not all((pseudo[ix]==k).any() for k in (0,1)):
        return boost,'Pseudo-labels H/A insuffisants : fallback géométrique'
    cols=['nearest_dist_diff','home_nearest_ball_dist','away_nearest_ball_dist','closest_distance',
          'home_mean_speed','away_mean_speed','ball_speed','tag_attack_home','tag_attack_away',
          'tag_setpiece','home_players_within_2m','away_players_within_2m',
          'home_players_within_5m','away_players_within_5m',
          'home_second_nearest_ball_dist','away_second_nearest_ball_dist','ball_gap_quality']
    X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy(np.float32)
    model=HistGradientBoostingClassifier(max_iter=85,max_leaf_nodes=15,
        learning_rate=.08,l2_regularization=1.5,random_state=2027)
    model.fit(X[ix],pseudo[ix])
    prob=np.full((n,3),1e-5,dtype=float)
    prob[:,model.classes_.astype(int)]=model.predict_proba(X)
    prob/=prob.sum(axis=1,keepdims=True)
    odds=np.clip(np.log(np.maximum(prob[:,0],1e-6)/np.maximum(prob[:,1],1e-6)),-3,3)
    boost[:,0]=+.5*odds;boost[:,1]=-.5*odds
    return boost,f'pseudo-labels automatiques={len(ix)} (sans GT)'

# ================== LES 6 FAMILLES FRONTIER ==================
XC03_MODELS=['baseline_hmm','graph_geometry','pseudo_distillation',
             'adaptive_multiscale','hybrid_frontier','hybrid_strong']

def xc03_predict_all(m):
    df=m['features'];blocks=m['blocks']
    base=make_logits(df)
    graph=graph_context_logits(df,base,pressure_weight=.55,imputation_weight=.75)
    sigma=(XC03_FPS if fps is None else fps)/XC03_REFERENCE_FPS
    multi=multiscale_logits(graph,blocks=blocks,
        scales=tuple(sigma*x for x in (3,15,50)),weights=(.35,.40,.25))
    dynamic=transition_cues(df)
    boost,status=xc03_pseudo_boost(df,graph,blocks)
    hybrid=.35*graph+.65*multi+.25*boost
    # Frontier original: graph+.6*boost; hybride: .35*graph+.65*multi+.25*boost.
    out={
        'baseline_hmm':xc03_decode(base,blocks,(13.,10.,7.)),
        'graph_geometry':xc03_decode(graph,blocks,(14.,10.,8.)),
        'pseudo_distillation':xc03_decode(graph+.6*boost,blocks,(14.,10.,8.)),
        'adaptive_multiscale':xc03_decode(multi,blocks,(14.,10.,8.),dynamic),
        'hybrid_frontier':xc03_decode(hybrid,blocks,(14.,10.,8.),dynamic),
        'hybrid_strong':xc03_decode(hybrid,blocks,(23.,17.,11.),dynamic),
    }
    return out,status

# ================== METRIQUES H/A SUR LABELS CONNUS ==================
def xc03_segments(array,blocks):
    result=[]
    for lo,hi in blocks:
        v=array[lo:hi+1]
        cuts=np.r_[0,np.flatnonzero(v[1:]!=v[:-1])+1,len(v)]
        for a,b in zip(cuts[:-1],cuts[1:]):
            if v[a] in ('H','A'):result.append((lo+int(a),lo+int(b),v[a]))
    return result

def xc03_metrics(truth,pred,period,frame):
    mask=np.isin(truth,['H','A'])
    cuts=np.r_[0,np.flatnonzero((mask[1:]!=mask[:-1]) |
           (period[1:]!=period[:-1]) | (frame[1:]!=frame[:-1]+1))+1,len(truth)]
    blocks=[(int(a),int(b-1)) for a,b in zip(cuts[:-1],cuts[1:]) if mask[a]]
    A=xc03_segments(truth,blocks);B=xc03_segments(pred,blocks)
    pairs=[];i=j=0
    while i<len(A) and j<len(B):
        sa,ea,ca=A[i];sb,eb,cb=B[j]
        ov=max(0,min(ea,eb)-max(sa,sb))
        if ov and ca==cb:pairs.append((ov/(max(ea,eb)-min(sa,sb)),i,j))
        if ea<=eb:i+=1
        if eb<=ea:j+=1
    pairs.sort(reverse=True)
    vals=[]
    for th in (.3,.4,.5,.6,.7):
        seen_a=set();seen_b=set();tp=0
        for iou,ii,jj in pairs:
            if iou<th:break
            if ii not in seen_a and jj not in seen_b:
                tp+=1;seen_a.add(ii);seen_b.add(jj)
        vals.append(2*tp/(len(A)+len(B)) if (len(A)+len(B)) else 0.0)
    yt=truth[mask];yp=pred[mask]
    f1=[2*np.sum((yt==c)&(yp==c))/max(1,int(np.sum(yt==c)+np.sum(yp==c))) for c in ('H','A')]
    return dict(proxy_spellF1=float(np.mean(vals)),HA_macroF1=float(np.mean(f1)),
        F1_H=float(f1[0]),F1_A=float(f1[1]),
        D_on_known_HA_pct=100*float(np.mean(yp=='D')),
        coverage_labeled=float(mask.mean()),n_labeled_blocks=len(blocks),
        possessions_pred=len(B),possessions_gt=len(A))

In [ ]:
# Un test de transportabilité : les modèles avec remise de jeu « Team » spécifique
# à Metrica sont exclus du classement externe.
ELIGIBLE_HA=['baseline_hmm','switch_strong','temperature3_strong','control_strong',
             'control_duel_strong','control_touch_strong','graph_geometry',
             'adaptive_multiscale','fusion_control_multiscale','fusion_control_postprocess']
EXPERIMENTAL_D=['D_V2_strong','fusion_D_V2_strong']
NON_ELIGIBLE=['nearest','control_restart_strong']+EXPERIMENTAL_D

def external_predict(m):
    obs=controller_variants(m['features'],m['blocks'],XC03_FPS,events=None)
    d,parts,flags=xc5_d_evidence(m['features'],m['blocks'],XC03_FPS)
    p=predict_specs(model_specs(obs,dscore=d,allow_restart=False),m['blocks'],XC03_FPS,target=False)
    return p,obs,parts

EXTERNAL_ROWS=[];EXTERNAL_AUDIT=[]
for mid in DEV_IDS:
    tic=time.time();tracking,metadata=xc03_source(mid)
    m=xc03_load(tracking,metadata)
    predictions,obs,dpart=external_predict(m)
    for name,pred in predictions.items():
        met=xc03_metrics(m['y'],pred,m['period'],m['frame'])
        EXTERNAL_ROWS.append({'match_id':mid,'modele':name,**met})
    EXTERNAL_AUDIT.append({'match_id':mid,'frames':len(m['y']),
                           'labeled_HA_frac':float(np.isin(m['y'],['H','A']).mean()),
                           'controller_direct_frac':float(np.mean(obs['detail'].controller_observed)),
                           'duel_frac':float(obs['detail'].duel.mean()),
                           'touch_candidate_frac':float(obs['detail'].touch_candidate.mean())})
    print('SkillCorner',mid,'frames',len(m['y']),'| temps',round(time.time()-tic,1),'s')
    del m,predictions,obs,dpart;gc.collect()
EXTERNAL_SCORES=pd.DataFrame(EXTERNAL_ROWS)
assert set(DEV_IDS)==set(EXTERNAL_SCORES.match_id.unique())
display(pd.DataFrame(EXTERNAL_AUDIT).round(4))
EXTERNAL_SCORES.to_csv(OUT/'XC_final_CV_par_match.csv',index=False)

### 7.1 Leaderboard externe et validation *leave-one-match-out*

Un score unique peut cacher un match exceptionnel. Nous affichons la moyenne **par match**, l'écart-type et les cinq résultats séparés. Le critère principal `robust = mean(proxy_spellF1) − 0.25 × std` a été fixé avant lecture des résultats. Nous évaluons également le macroF1 H/A, la fragmentation et les faux `D` sur H/A connu.

L'ablation est lisible : comparer `switch_strong → control_strong → control_duel_strong → control_touch_strong → fusion_control_multiscale`, sans attribuer un gain de TabICL à une variante non entraînée.

In [ ]:
CV_SUM=EXTERNAL_SCORES.groupby('modele').agg(
 matches=('match_id','nunique'),spellF1_mean=('proxy_spellF1','mean'),
 spellF1_std=('proxy_spellF1','std'),HA_macroF1=('HA_macroF1','mean'),
 F1_H=('F1_H','mean'),F1_A=('F1_A','mean'),
 false_D_HA_pct=('D_on_known_HA_pct','mean'),
 n_spells=('possessions_pred','mean')).reset_index()
CV_SUM['robust']=CV_SUM.spellF1_mean-.25*CV_SUM.spellF1_std.fillna(0)
CV_SUM['eligible_HA']=CV_SUM.modele.isin(ELIGIBLE_HA)
CV_SUM['D_validated']=False
CV_SUM=CV_SUM.sort_values(['robust','HA_macroF1'],ascending=False)
print('LEADERBOARD EXTERNE — scores H/A uniquement :')
display(CV_SUM.round(4))
heat=EXTERNAL_SCORES.pivot(index='modele',columns='match_id',values='proxy_spellF1').reindex(CV_SUM.modele)
fig,ax=plt.subplots(figsize=(11,7));im=ax.imshow(heat.values,cmap='viridis',aspect='auto',vmin=0,vmax=1)
ax.set_xticks(range(len(heat.columns)),list(map(str,heat.columns)),rotation=45,ha='right')
ax.set_yticks(range(len(heat.index)),list(heat.index))
for i in range(len(heat.index)):
 for j in range(len(heat.columns)):
  ax.text(j,i,f'{heat.iat[i,j]:.3f}',ha='center',va='center',color='white',fontsize=8)
fig.colorbar(im,ax=ax,label='Proxy spellF1 H/A');ax.set_title('Ablations sur 5 matchs SkillCorner');plt.tight_layout();plt.show()
CV_SUM.to_csv(OUT/'XC_final_CV_leaderboard.csv',index=False)

def choose_from_cv(scores,names):
    d=scores[scores.modele.isin(names)].groupby('modele').agg(
          avg=('proxy_spellF1','mean'),sd=('proxy_spellF1','std'),macro=('HA_macroF1','mean')).fillna(0)
    if d.empty:raise RuntimeError('Aucun modèle éligible au classement')
    d['criterion']=d.avg-.25*d.sd
    return str(d.sort_values(['criterion','macro'],ascending=False).index[0])

folds=[]
for held in DEV_IDS:
    train=EXTERNAL_SCORES[EXTERNAL_SCORES.match_id!=held]
    selected=choose_from_cv(train,ELIGIBLE_HA)
    test=EXTERNAL_SCORES[(EXTERNAL_SCORES.match_id==held)&(EXTERNAL_SCORES.modele==selected)].iloc[0]
    folds.append({'held_out_match':held,'selected_without_match':selected,
                  'proxy_spellF1':test.proxy_spellF1,'HA_macroF1':test.HA_macroF1})
LOMO=pd.DataFrame(folds)
print('Leave-one-match-out :');display(LOMO.round(4))
LOMO.to_csv(OUT/'XC_final_LOMO.csv',index=False)
BASE_LOCKED=choose_from_cv(EXTERNAL_SCORES,ELIGIBLE_HA)
print('CANDIDAT PRINCIPAL H/A (sans TabICL) :',BASE_LOCKED)

## 8 · Tabular Foundation Model : TabICLv2 + boosting classique

**Hypothèse :** une représentation tabulaire non linéaire peut mieux combiner les distances, la visibilité et le mouvement collectif que les scores heuristiques. Nous comparons **TabICLv2**, un modèle de fondation tabulaire préentraîné avec interface scikit-learn (`fit`, `predict_proba`), à `HistGradientBoosting` et à une règle physique.

**Sémantique importante :** TabICLv2 ne connaît que les classes H/A des **matchs externes** de développement ; il ne produit pas une probabilité de jeu arrêté. La probabilité `p(H)`/`p(A)` vient en appoint des logits H/A, le score D est inchangé. `fit` fournit le contexte d'exemples au modèle préentraîné ; l'inférence est locale dans Kaggle. Les poids publics pourront être téléchargés ; **aucune donnée privée du concours n'est envoyée à un serveur d'inférence**.

**Contraintes du test :** l'échantillonnage du *screening* est équilibré et ne donne **pas** un spellF1 temporel. Pour entrer au leaderboard final, TabICL devra passer une CV temporelle **sur cinq matchs entiers**, avec la même segmentation que les modèles précédents. Les résultats de screening restent diagnostiques jusqu'à cette validation.

In [ ]:
TFM_FEATURES=['ball_visible','home_nearest_ball_dist','away_nearest_ball_dist',
 'nearest_dist_diff','closest_distance','home_second_nearest_ball_dist',
 'away_second_nearest_ball_dist','home_players_within_2m','away_players_within_2m',
 'home_players_within_5m','away_players_within_5m','home_mean_speed','away_mean_speed',
 'ball_gap_quality','home_imputed_nearest','away_imputed_nearest']

def tfm_X(df):
    # Exclusivement des features communes. Ni labels, ni frame, ni identifiant de match,
    # ni événements Metrica indisponibles au test SkillCorner.
    x={col:(pd.to_numeric(df[col],errors='coerce') if col in df else
            pd.Series(np.nan,index=df.index,dtype=float)) for col in TFM_FEATURES}
    return pd.DataFrame(x).replace([np.inf,-np.inf],np.nan).astype(np.float32)

def sample_ha(y,k,seed):
    rng=np.random.default_rng(seed)
    res=[]
    for lab in ('H','A'):
        ids=np.flatnonzero(np.asarray(y)==lab)
        if len(ids):res.extend(rng.choice(ids,size=min(len(ids),k//2),replace=False).tolist())
    rng.shuffle(res)
    return np.asarray(res,int)

TFM_OK=False;TFM_ERROR=''
if RUN_TABICL_SCREEN or RUN_TABICL_FULL_CV:
    try:
        if importlib.util.find_spec('tabicl') is None:
            print('Installation locale de tabicl depuis PyPI ...')
            ret=subprocess.run([sys.executable,'-m','pip','install','-q','tabicl'],capture_output=True,text=True)
            if ret.returncode:raise RuntimeError(ret.stderr[-1000:])
        from tabicl import TabICLClassifier
        TFM_OK=True
        print('TabICLv2 importé, checkpoint officiel au premier appel.')
    except Exception as e:
        TFM_ERROR=str(e);print('TabICLv2 indisponible :',TFM_ERROR[:400])

### 8.1 Évaluation rapide H/A par match (vérification du foundation model)

Pour chaque fold : 4 matchs fournissent le contexte d'entraînement, 1 match fournit l'évaluation. Le sous-échantillonnage se fait **par équipe et par match**, jamais aléatoirement en mélangeant les frames d'un match entre train et validation. Ce premier résultat n'est pas le classement final ; il sert à savoir si le modèle tabulaire fonctionne et apporte un signal géométrique utilisable.

In [ ]:
TFM_POOL={};TFM_SCREEN=[];TFM_ERRORS=[]
if RUN_TABICL_SCREEN:
    for mid in DEV_IDS:
        t0=time.time();p,q=xc03_source(mid);match=xc03_load(p,q)
        X=tfm_X(match['features']); y=match['y']
        train_i=sample_ha(y,TABICL_TRAIN_PER_GAME,SEED+mid)
        val_i=sample_ha(y,TABICL_VAL_PER_GAME,SEED+mid+1)
        TFM_POOL[mid]={'Xtr':X.iloc[train_i].reset_index(drop=True),'ytr':y[train_i],
                       'Xval':X.iloc[val_i].reset_index(drop=True),'yval':y[val_i]}
        print('Pool',mid,'train',len(train_i),'eval',len(val_i),'temps',round(time.time()-t0,1))
        del match,X;gc.collect()
    for held in DEV_IDS:
        training=[m for m in DEV_IDS if m!=held]
        Xtrain=pd.concat([TFM_POOL[m]['Xtr'] for m in training],ignore_index=True)
        ytrain=np.concatenate([TFM_POOL[m]['ytr'] for m in training])
        Xval=TFM_POOL[held]['Xval'];yval=TFM_POOL[held]['yval']
        diff=Xval.nearest_dist_diff.to_numpy(float)
        phys=np.where(np.nan_to_num(diff,nan=-1)<0,'H','A')
        TFM_SCREEN.append({'held':held,'modele':'nearest_physics','HA_macroF1':f1_score(yval,phys,labels=['H','A'],average='macro',zero_division=0)})
        gb=HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=15,random_state=SEED)
        gb.fit(Xtrain,ytrain)
        TFM_SCREEN.append({'held':held,'modele':'HistGradientBoosting','HA_macroF1':f1_score(yval,gb.predict(Xval),labels=['H','A'],average='macro',zero_division=0)})
        if TFM_OK:
            try:
                model=TabICLClassifier(n_estimators=2,random_state=SEED)
                model.fit(Xtrain,ytrain)
                pred=model.predict(Xval)
                TFM_SCREEN.append({'held':held,'modele':'TabICLv2','HA_macroF1':f1_score(yval,pred,labels=['H','A'],average='macro',zero_division=0)})
            except Exception as e:
                TFM_ERRORS.append({'held':held,'message':str(e)[:500]})
                print('TabICL erreur sur match',held,':',str(e)[:150])
        print('Fold TFM screening',held,'terminé')
        del gb,Xtrain,ytrain;gc.collect()
TFM_SCREEN=pd.DataFrame(TFM_SCREEN)
if not TFM_SCREEN.empty:
    display(TFM_SCREEN.pivot(index='modele',columns='held',values='HA_macroF1').round(4))
    TFM_SCREEN.to_csv(OUT/'XC_final_TabICL_screening_HA.csv',index=False)
if TFM_ERRORS:display(pd.DataFrame(TFM_ERRORS))

### 8.2 Fusion TabICL avec Viterbi et **validation temporelle complète**

Pour une probabilité `P(H)` et `P(A)`, nous ajoutons aux logits `H/A` un log-ratio `log(P(H)/P(A))`, pondéré par `γ=0.35` fixé à l'avance. **Le score D n'est pas modifié**. Nous décodons ensuite avec `switch_strong` sur le match complet. 

Le code peut exécuter cinq folds temporels complets (`RUN_TABICL_FULL_CV=True` dans la configuration). Cette étape demande davantage de temps/VRAM ; tant qu'elle n'est pas terminée sur **les cinq matchs**, `TabICLv2_fusion` est explicitement **ineligible** à la sélection finale. Aucun score de screening ne sera substitué au spellF1.

In [ ]:
TFM_GAMMA=.35

def fusion_tabicl(z,p,classes,gamma=TFM_GAMMA):
    classes=np.asarray(classes).astype(str)
    assert set(classes)=={'H','A'}
    H=int(np.where(classes=='H')[0][0]);A=int(np.where(classes=='A')[0][0])
    odds=np.log(np.maximum(p[:,H],1e-5))-np.log(np.maximum(p[:,A],1e-5))
    odds=np.clip(odds,-4,4)
    out=np.array(z,copy=True)
    out[:,0]+=gamma*odds/2
    out[:,1]-=gamma*odds/2
    return out

_test=fusion_tabicl(np.zeros((2,3)),np.array([[.8,.2],[.3,.7]]),['H','A'])
assert (_test[:,2]==0).all(),'La fusion doit laisser D inchangé'
TFM_FULL=[]
if RUN_TABICL_FULL_CV:
    if not TFM_OK:raise RuntimeError('La CV complète requiert TabICLv2 installé localement')
    if not TFM_POOL:raise RuntimeError('Activer aussi RUN_TABICL_SCREEN pour créer le pool externe')
    for held in DEV_IDS:
        train=[m for m in DEV_IDS if m!=held]
        Xtrain=pd.concat([TFM_POOL[m]['Xtr'] for m in train],ignore_index=True)
        ytrain=np.concatenate([TFM_POOL[m]['ytr'] for m in train])
        p,q=xc03_source(held);m=xc03_load(p,q)
        model=TabICLClassifier(n_estimators=2,random_state=SEED)
        model.fit(Xtrain,ytrain)
        xv=tfm_X(m['features'])
        probs=[]
        for a in range(0,len(xv),TABICL_QUERY_BATCH):
            probs.append(model.predict_proba(xv.iloc[a:a+TABICL_QUERY_BATCH]))
        prob=np.vstack(probs)
        fused=fusion_tabicl(make_logits(m['features']),prob,model.classes_)
        pred=xc03_decode(fused,m['blocks'],PEN_STRONG)
        met=xc03_metrics(m['y'],pred,m['period'],m['frame'])
        TFM_FULL.append({'match_id':held,'modele':'TabICLv2_fusion',**met})
        print('TFM temporel',held,'spellF1',round(met['proxy_spellF1'],4))
        del m,model,pred,probs,prob,xv;gc.collect()
TFM_FULL=pd.DataFrame(TFM_FULL)
if len(TFM_FULL)==len(DEV_IDS):
    display(TFM_FULL.round(4))
    TFM_FULL.to_csv(OUT/'XC_final_TabICL_fullCV_spells.csv',index=False)
else:
    print('TabICLv2_fusion n’est PAS éligible au classement final : validation temporelle incomplète.')

## 9 · Leaderboard final sur données externes et choix H/A

Nous joignons TabICL au classement **seulement** lorsque ses cinq folds temporels complets sont valides. Les modèles `D_V2_*` restent visibles comme diagnostics, mais aucun SkillCorner H/A ne peut valider leur rappel d'arrêts. Le classement des candidats H/A est donc une **sélection de la composante possession**, pas une garantie de performance complète sur H/A/D.

Nous verrouillons le choix **avant** le test réservé et avant la vérification publique. Le classement expose la moyenne, la dispersion et les faux `D`. Le modèle exportable peut être choisi explicitement dans la dernière cellule, qui vérifie son admissibilité.

In [ ]:
ALL_EXTERNAL=EXTERNAL_SCORES.copy()
if len(TFM_FULL)==len(DEV_IDS):
    ALL_EXTERNAL=pd.concat([ALL_EXTERNAL,TFM_FULL],ignore_index=True)
ALL_ELIGIBLE=ELIGIBLE_HA+(['TabICLv2_fusion'] if len(TFM_FULL)==len(DEV_IDS) else [])
FINAL_LEADERBOARD=ALL_EXTERNAL.groupby('modele').agg(
    matches=('match_id','nunique'),spellF1_mean=('proxy_spellF1','mean'),
    spellF1_std=('proxy_spellF1','std'),HA_macroF1=('HA_macroF1','mean'),
    false_D_HA_pct=('D_on_known_HA_pct','mean'),
    predicted_spells=('possessions_pred','mean')).reset_index()
FINAL_LEADERBOARD['robust']=FINAL_LEADERBOARD.spellF1_mean-.25*FINAL_LEADERBOARD.spellF1_std.fillna(0)
FINAL_LEADERBOARD['eligible']=FINAL_LEADERBOARD.modele.isin(ALL_ELIGIBLE)
FINAL_LEADERBOARD['D_validated']=False
FINAL_LEADERBOARD=FINAL_LEADERBOARD.sort_values(['robust','HA_macroF1'],ascending=False).reset_index(drop=True)
print('LEADERBOARD FINAL — cinq matchs EXTERNES / H/A uniquement :')
display(FINAL_LEADERBOARD.round(4))
FINAL_LEADERBOARD.to_csv(OUT/'XC_FINAL_LEADERBOARD.csv',index=False)
LOCKED_HA=choose_from_cv(ALL_EXTERNAL,ALL_ELIGIBLE)
print('CANDIDAT VERROUILLÉ SUR CV EXTERNE :',LOCKED_HA)

## 10 · Vérification finale sur un match externe réservé

Le match externe `2017461` n'a pas été utilisé pour construire ni choisir les variantes. Nous l'évaluons **une seule fois**, uniquement après le verrouillage. Si le téléchargement est impossible, le rapport l'indique ; aucun score synthétique ne le remplace. Après sa lecture, nous ne réajustons plus les constantes. Un ancien test sur `2011166` ne peut pas jouer de nouveau le rôle d'un holdout indépendant.

In [ ]:
HOLDOUT=None
if RUN_FRESH_HOLDOUT_ONCE:
    try:
        p,q=xc03_source(FRESH_HOLDOUT);m=xc03_load(p,q)
        if LOCKED_HA=='TabICLv2_fusion':
            tx=pd.concat([TFM_POOL[k]['Xtr'] for k in DEV_IDS],ignore_index=True)
            ty=np.concatenate([TFM_POOL[k]['ytr'] for k in DEV_IDS])
            model=TabICLClassifier(n_estimators=2,random_state=SEED)
            model.fit(tx,ty)
            x=tfm_X(m['features']);probs=np.vstack([
                  model.predict_proba(x.iloc[a:a+TABICL_QUERY_BATCH])
                  for a in range(0,len(x),TABICL_QUERY_BATCH)])
            z=fusion_tabicl(make_logits(m['features']),probs,model.classes_)
            pred=xc03_decode(z,m['blocks'],PEN_STRONG)
        else:
            ps,obs,_=external_predict(m);pred=ps[LOCKED_HA]
        HOLDOUT={'match_id':FRESH_HOLDOUT,'model_locked':LOCKED_HA,
                 **xc03_metrics(m['y'],pred,m['period'],m['frame'])}
        display(pd.DataFrame([HOLDOUT]).round(4))
        pd.DataFrame([HOLDOUT]).to_csv(OUT/'XC_final_holdout_une_fois.csv',index=False)
        del m,pred;gc.collect()
    except Exception as e:
        print('TEST EXTERNE INDISPONIBLE :',str(e)[:500])
        print('Aucun résultat substitué ; candidate CV conservé.')

## 11 · Audit géométrique, limites et conformité

Les règles footballistiques restantes ne doivent pas devenir des approximations non documentées :

- **Homographie** : sans correspondances pixels ↔ terrain fiables ou erreurs de reprojection fournies, nous ne pouvons pas calculer un vrai score de reprojection. Nous vérifions la cohérence des coordonnées en mètres et le taux de points hors terrain, sans inventer de confiance.
- **Vidéo broadcast** : la synchronisation avec le tracking n'est pas garantie ; nous n'y extrayons pas de labels automatiques non validés.
- **Smart Tagging `DEFENDING`** : le champ `Team` est audité, pas inversé par hypothèse.
- **Micro-possessions** : un post-traitement n'est accepté que s'il progresse sur les matchs externes.
- **Arrêts D** : aucun score d'exactitude complète n'est revendiqué sans GT `D` externe. La sélection automatique retient H/A ; la branche D V2 reste explicitement expérimentale.

In [ ]:
assert len(TARGET_PREDS['switch_strong'])==N_FRAMES
assert all(np.isin(v,['H','A','D','X']).all() for v in TARGET_PREDS.values())
assert all((v[HALVES[1][0]:SECOND_KICKOFF]=='D').all() for v in TARGET_PREDS.values())
assert all((v[:HALVES[0][0]]=='X').all() and (v[HALVES[1][1]+1:]=='X').all() for v in TARGET_PREDS.values())
assert np.isfinite(base_evid['base']).all()
assert len(DEV_IDS)==5 and FRESH_HOLDOUT not in DEV_IDS
# tests unitaires: non création de contrôle lors d'une occlusion isolée ; reset des remises en jeu
_toy=pd.DataFrame({'frame':np.arange(60),'ball_visible':np.r_[np.ones(15),np.zeros(45)],
 'ball_x':np.r_[np.full(15,.5),np.full(45,np.nan)],
 'ball_y':np.r_[np.full(15,.5),np.full(45,np.nan)],
 'home_nearest_ball_dist':np.r_[np.full(15,1.),np.full(45,np.nan)],
 'away_nearest_ball_dist':np.full(60,4.),
 'home_players_within_2m':np.r_[np.ones(15),np.full(45,np.nan)],
 'away_players_within_2m':np.zeros(60)})
_strength,_state,_=controller_evidence(_toy,[(0,59)],25,events=None)
assert not _state.controller_observed.to_numpy(bool)[15:].any()
assert (_state.last_controller.to_numpy()[15:59]==0).all() # propagation mémorisée, sans nouveau contrôle
# Test du reset après cinq frames de contrôle établi
_reset=[{'lo':20,'hi':25,'code':'SET PIECES','type':'Free Kick','side':'H'}]
_,_after,_=controller_evidence(_toy,[(0,59)],25,events=_reset)
assert (_after.last_controller.to_numpy()[25:]==-1).all()
print('TESTS : alignement, alphabet, kickoff D, X, blocs, occlusion, reset, fusion H/A OK.')
qc={'median_ball_visibility':float(features.ball_visible.mean()),
    'outside_ball_observations_pct':float(100*np.mean(D_FLAGS.outside_confirmed[MASK_FLEX])),
    'tagged_restarts':int(len(base_evid['types'])),'n_candidate_models':len(TARGET_PREDS)}
display(pd.DataFrame([qc]).round(3))

## 12 · Vérification publique du **candidat verrouillé seulement**

Cette étape est déclenchée **après** le choix H/A externe. Elle contrôle uniquement le format et l'alignement sur les cinq minutes publiées. Nous ne calculons pas de classement des variantes sur cette GT : un tel classement conduirait à du tuning interdit. Si le fichier public n'est pas présent dans `BAR-DATA`, le notebook conserve la validation stricte de format et indique l'absence de la vérification facultative.

In [ ]:
# Production de la prédiction TabICL cible si elle a été sélectionnée avec une CV complète.
if LOCKED_HA=='TabICLv2_fusion' and 'TabICLv2_fusion' not in TARGET_PREDS:
    trX=pd.concat([TFM_POOL[mid]['Xtr'] for mid in DEV_IDS],ignore_index=True)
    trY=np.concatenate([TFM_POOL[mid]['ytr'] for mid in DEV_IDS])
    model=TabICLClassifier(n_estimators=2,random_state=SEED);model.fit(trX,trY)
    Xtarget=tfm_X(features)
    p=np.vstack([model.predict_proba(Xtarget.iloc[a:a+TABICL_QUERY_BATCH])
                for a in range(0,N_FRAMES,TABICL_QUERY_BATCH)])
    z=fusion_tabicl(make_logits(features),p,model.classes_)
    TARGET_PREDS['TabICLv2_fusion']=enforce_published_blocks(viterbi(z,penalties=PEN_STRONG))

if RUN_PUBLIC_PIPELINE_CHECK:
    files=list(Path('/kaggle/input').rglob('BAR-ATH_possession_gt_public.csv')) if Path('/kaggle/input').exists() else []
    if files:
        public=pd.read_csv(files[0]);assert len(public)==N_FRAMES
        assert {'frame','label'}.issubset(public.columns)
        assert np.array_equal(public.frame.to_numpy(),np.arange(N_FRAMES))
        truth=public.label.astype(str).to_numpy()
        valid=truth!='X';candidate=TARGET_PREDS[LOCKED_HA]
        print('PUBLIC PIPELINE CHECK:',int(valid.sum()),'frames, candidat déjà verrouillé',LOCKED_HA)
        display(pd.crosstab(pd.Categorical(truth[valid],categories=['H','A','D']),
                    pd.Categorical(candidate[valid],categories=['H','A','D','X']),dropna=False))
        print('Contrôle uniquement : aucun seuil ni choix ne dépend de ce tableau.')
    else:
        print('GT publique non attachée : ce contrôle facultatif est sauté.')

## 13 · Traçabilité et livraison : choix du modèle dans la **dernière cellule**

Nous conservons les tableaux CSV de validation, la matrice de désaccord, les tests de cohérence et le protocole de sélection. La soumission **doit être régénérable** en exécutant ce notebook depuis les données de la compétition. Aucun tracking, événement ou extrait vidéo privé ne doit être ajouté au dépôt public.

### Sélection de l'équipe Xcelerate

- `MODEL_TO_EXPORT='AUTO'` exporte le candidat issu du leaderboard **H/A externe**, jamais des cinq minutes publiques.
- `MODEL_TO_EXPORT='switch_strong'` (ou un autre nom éligible) permet à l'équipe de choisir explicitement un candidat documenté.
- Une variante `D_V2_*` n'est **pas** considérée validée par SkillCorner : l'export manuel expérimental nécessite d'activer volontairement `ALLOW_UNVALIDATED_D=True` et de conserver cet avertissement dans le manifeste.
- La dernière cellule écrit un unique CSV nommé `Xcelerate.csv`, **199 319 lignes**, colonnes `frame,label`, et affiche son checksum SHA256.

La décision complète H/A/D devra être revue avant dépôt si la qualité de `D` n'a pas été validée sur un dataset externe avec de vrais arrêts de jeu.

In [ ]:
# ==============================================================
# XCELERATE — DERNIÈRE CELLULE : CHOIX / LEADERBOARD / SOUMISSION
# ==============================================================
MODEL_TO_EXPORT='AUTO'  # 'AUTO', 'switch_strong', 'control_strong', 'fusion_control_multiscale', ...
ALLOW_UNVALIDATED_D=False  # autorisation EXPERIMENTALE explicite pour les modèles D_V2

print('LEADERBOARD H/A SUR CINQ MATCHS EXTERNES')
display(FINAL_LEADERBOARD[['modele','matches','spellF1_mean','spellF1_std','HA_macroF1',
    'false_D_HA_pct','robust','eligible','D_validated']].round(4))

SELECTED=LOCKED_HA if MODEL_TO_EXPORT.upper()=='AUTO' else MODEL_TO_EXPORT
if SELECTED not in TARGET_PREDS:
    raise ValueError('Modèle inconnu: '+SELECTED+' | disponibles: '+', '.join(TARGET_PREDS))
if SELECTED not in ALL_ELIGIBLE:
    if SELECTED in EXPERIMENTAL_D and ALLOW_UNVALIDATED_D:
        print('ATTENTION : branche D V2 non validée avec de vrais labels D externes.')
    else:
        raise ValueError('Modèle non éligible sur données externes; sélection bloquée : '+SELECTED)
assert N_FRAMES==199319
candidate=np.asarray(TARGET_PREDS[SELECTED],dtype='<U1')
if len(candidate)!=N_FRAMES or not np.isin(candidate,['H','A','D','X']).all():
    raise ValueError('Prédiction invalide')
candidate=enforce_published_blocks(candidate)
assert np.all(candidate[:HALVES[0][0]]=='X')
assert np.all(candidate[HALVES[1][0]:SECOND_KICKOFF]=='D')
assert np.all(candidate[HALVES[1][1]+1:]=='X')
filename=OUT/'Xcelerate.csv'
final=pd.DataFrame({'frame':np.arange(N_FRAMES,dtype=np.int32),'label':candidate})
assert final.notna().all().all() and np.array_equal(final.frame.to_numpy(),np.arange(N_FRAMES))
final.to_csv(filename,index=False)
sha=hashlib.sha256(filename.read_bytes()).hexdigest()
manifest={'team':'Xcelerate','model_exported':SELECTED,'model_locked_by_external_CV':LOCKED_HA,
          'source_of_selection':'SkillCorner 5 independent matches (H/A proxy spellF1)',
          'external_development_matches':DEV_IDS,'fresh_holdout':HOLDOUT,
          'D_status':'EXPERIMENTAL: external D recall NOT validated',
          'no_competition_GT_training_or_tuning':True,'n_frames':N_FRAMES,
          'csv_sha256':sha,'model_available_in_notebook':True,'file':str(filename)}
(OUT/'Xcelerate_selection_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf8')
print('\nMODELE EXPORTÉ :',SELECTED)
print('CSV :',filename,'| lignes :',len(final),'| SHA256 :',sha)
print('H/A/D/X :',final.label.value_counts().to_dict())
print('VALIDATION FORMAT : OK — vérifier aussi avec validate_submission.py officiel si disponible.')
print('ATTENTION : qualité D non indépendante ; ne pas publier les données privées de match.')
display(final.iloc[[0,11184,11185,90651,90652,110807,110808,113741,113742,190197,190198,199318]])

In [ ]:
# XCELERATE FC — CELLULE A / Rapport des résultats et audit du CSV
# À exécuter APRÈS la dernière cellule de sélection du notebook intégral.
import os, json, hashlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

XC_R_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
XC_R_DIR.mkdir(parents=True, exist_ok=True)
REPORT = []
def sec(title, value=None):
    REPORT.append('\n' + '='*23 + ' ' + title + ' ' + '='*23)
    if value is not None:
        REPORT.append(str(value))
def fmt(obj, rows=45, cols=18, digits=4):
    if obj is None:
        return '(pas de valeur)'
    if isinstance(obj, pd.DataFrame):
        if obj.empty: return '(table vide)'
        with pd.option_context('display.max_rows', rows, 'display.max_columns', cols,
                               'display.width', 220, 'display.max_colwidth', 40):
            return obj.round(digits).to_string(index=False)
    return str(obj)

def val(name, default='NON_DISPONIBLE'):
    return globals().get(name, default)

sec('XCELERATE FC — RAPPORT TECHNIQUE POUR REDACTION')
sec('0. Configuration et protocole')
for key in ['SEED','DEV_IDS','FRESH_HOLDOUT','PEN_BASE','PEN_STRONG',
            'PEN_FRONTIER','R_CTRL','CTRL_MIN_LEN','CTRL_MAX_AGE_SEC',
            'CTRL_MAX_BALL_SPEED','TOUCH_ACCEL','RUN_TABICL_SCREEN',
            'RUN_TABICL_FULL_CV','RUN_FRESH_HOLDOUT_ONCE',
            'BASE_LOCKED','LOCKED_HA','MODEL_TO_EXPORT','SELECTED',
            'ALLOW_UNVALIDATED_D']:
    REPORT.append(f'{key}: {val(key)}')
REPORT.append('Evaluation SkillCorner = autres matchs externes, labels H/A uniquement.')
REPORT.append('La GT publique de 5 min est exclue de toute selection ou optimisation.')

sec('1. Leaderboard externe H/A : 5 matchs SkillCorner')
if isinstance(globals().get('FINAL_LEADERBOARD'),pd.DataFrame):
    x=FINAL_LEADERBOARD.copy()
    names=['modele','matches','spellF1_mean','spellF1_std','HA_macroF1',
           'false_D_HA_pct','predicted_spells','robust','eligible','D_validated']
    REPORT.append(fmt(x[[c for c in names if c in x.columns]],rows=50))
else:
    REPORT.append('FINAL_LEADERBOARD manquant : cellule 9 non executee ?')

sec('2. Ablations : resultats de chaque modele sur chaque match externe')
if isinstance(globals().get('ALL_EXTERNAL'),pd.DataFrame):
    x=ALL_EXTERNAL.copy()
    cols=['match_id','modele','proxy_spellF1','HA_macroF1','F1_H','F1_A',
          'D_on_known_HA_pct','possessions_pred']
    REPORT.append(fmt(x[[c for c in cols if c in x.columns]].sort_values(
        ['modele','match_id']),rows=110))
else:
    REPORT.append('ALL_EXTERNAL manquant.')

sec('3. Validation croisee leave-one-match-out')
REPORT.append(fmt(globals().get('LOMO')))
sec('4. Test sur le nouveau match externe reserve (une evaluation)')
REPORT.append(fmt(globals().get('HOLDOUT')))

sec('5. TabICLv2 : screening et validation temporelle')
REPORT.append('TFM_OK: '+str(val('TFM_OK')))
REPORT.append('TFM_ERRORS: '+str(val('TFM_ERRORS')))
REPORT.append('TFM_SCREEN (macroF1 H/A sur sous-echantillons):')
REPORT.append(fmt(globals().get('TFM_SCREEN')))
REPORT.append('TFM_FULL (spellF1 H/A temporel si active):')
REPORT.append(fmt(globals().get('TFM_FULL')))

sec('6. Diagnostics sans GT sur Barcelona–Athletic')
REPORT.append(fmt(globals().get('TARGET_DIAG')))
sec('7. Divergences versus switch_strong, 15 plus longues')
dis=globals().get('DISAGREEMENTS')
REPORT.append(fmt(dis.head(15) if isinstance(dis,pd.DataFrame) else dis))
sec('8. Tests et indicateurs de qualite des donnees')
if 'features' in globals():
    REPORT.append('n_features: '+str(len(features)))
    for col in ['ball_visible','ball_gap_quality']:
        if col in features.columns:
            a=pd.to_numeric(features[col],errors='coerce')
            REPORT.append(f'{col}: moyenne={float(a.mean()):.4f}, NaN={int(a.isna().sum())}')
if 'base_evid' in globals():
    REPORT.append('Tags de remise en jeu: '+str(len(base_evid.get('types',[]))))
if 'D_FLAGS' in globals():
    for col in ['outside_confirmed','prep_restart','restart','memory_stop','alive']:
        if hasattr(D_FLAGS,col):
            REPORT.append(f'{col}: {100*np.asarray(getattr(D_FLAGS,col),dtype=bool).mean():.3f}% des frames')

# Vérification du CSV à partir de son contenu réel, pas d'une prédiction en mémoire.
sec('9. CONTROLES DU CSV FINAL — dossier de soumission')
csv_path = XC_R_DIR/'Xcelerate.csv'
if not csv_path.is_file():
    for name in ['Xcelerate FC.csv','Xcelerate-FC.csv']:
        if (XC_R_DIR/name).exists():csv_path=XC_R_DIR/name;break
if not csv_path.is_file():
    REPORT.append('ERREUR: aucun Xcelerate.csv trouvé. Executer la derniere cellule du notebook.')
else:
    df=pd.read_csv(csv_path, dtype={'label':'object'})
    problems=[]
    if list(df.columns)!=['frame','label']:problems.append(f'colonnes={list(df.columns)}')
    if len(df)!=199319:problems.append(f'lignes={len(df)} != 199319')
    if 'frame' in df.columns and not np.array_equal(df.frame.to_numpy(),np.arange(199319)):
        problems.append('frames non contigus ou non tries')
    if 'label' in df.columns:
        p=df.label.to_numpy()
        if df.label.isna().any():problems.append('labels manquants')
        if not np.isin(p,['H','A','D','X']).all():problems.append('alphabet invalide')
        if len(p)==199319:
            if 'HALVES' in globals() and 'SECOND_KICKOFF' in globals():
                for lo,hi in HALVES:
                    if np.any(p[lo:hi+1]=='X'):problems.append(f'X dans plage notee {lo}:{hi}')
                if not np.all(p[HALVES[1][0]:SECOND_KICKOFF]=='D'):
                    problems.append('kickoff D impose incorrect')
            n_segments=int(np.sum(np.isin(p,['H','A']) & np.r_[True,p[1:]!=p[:-1]]))
            REPORT.append('Nombre de possessions H/A (video complete): '+str(n_segments))
            REPORT.append('Compte des labels: '+str(df.label.value_counts().to_dict()))
            REPORT.append('Labels aux bords: '+str([(i,p[i]) for i in
               [0,11184,11185,90651,90652,110807,110808,113741,113742,190197,190198,199318]]))
    sha=hashlib.sha256(csv_path.read_bytes()).hexdigest()
    REPORT.append('CSV source: '+str(csv_path))
    REPORT.append('CSV SHA256: '+sha)
    REPORT.append('CSV taille_octets: '+str(csv_path.stat().st_size))
    REPORT.append('CSV taille_Mo: '+f'{csv_path.stat().st_size/1e6:.3f}')
    REPORT.append('CONTROLES STRICTS: '+('OK' if not problems else 'ECHEC: '+' ; '.join(problems)))
    if csv_path.stat().st_size>10_000_000:
        REPORT.append('ALERTE: CSV >10MB, limite du formulaire.')
    if not problems:
        team_file=XC_R_DIR/'Xcelerate FC.csv' # nom exact de l'equipe selon le message
        if team_file.resolve()!=csv_path.resolve():shutil.copyfile(csv_path,team_file)
        REPORT.append('Fichier avec nom dequipe pour le formulaire: '+str(team_file))
        REPORT.append('SHA256 fichier equipe: '+hashlib.sha256(team_file.read_bytes()).hexdigest())

sec('10. Etat de la reproductibilite')
REPORT.append('Depot: https://github.com/khalifasimboro/Xcelerate-FC')
REPORT.append('Le script officiel validate_submission.py doit aussi valider le CSV exact.')
REPORT.append('Ne jamais publier le tracking, les tags, des extraits video ni des exports prives.')
REPORT.append('Le rapport doit preciser source des labels externes, quantites, hypotheses et limites D.')

content='\n'.join(REPORT)+'\n'
text_path=XC_R_DIR/'XCELERATE_RESULTATS_POUR_RAPPORT.txt'
text_path.write_text(content,encoding='utf-8')
print(content)
print('\nRAPPORT SAUVE:',text_path)
print('COPIER LE TEXTE CI-DESSUS ET ME L ENVOYER.')


In [ ]:
# XCELERATE FC — CELLULE B / Audit OFFICIEL de format et GT publique (NON SELECTIF)
# Ne change ni SELECTED, ni LOCKED_HA, ni les poids du modele.
import sys, importlib.util, urllib.request, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display

XC_O = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
selected = globals().get('SELECTED', globals().get('LOCKED_HA'))
assert selected in TARGET_PREDS, f'Candidat selectionne introuvable: {selected}'
print('MODELE VERROUILLE AVANT GT PUBLIQUE:',selected)

# Charger LA fonction de score officielle pour eviter un spellF1 approximatif.
# Kaggle: Internet ON au premier lancement, ou fichier possession_metrics.py ajoute comme dataset.
pm_path = XC_O / 'possession_metrics.py'
if not pm_path.exists():
    found = []
    for root in ['/kaggle/input','/kaggle/working']:
        if Path(root).exists():
            found.extend(Path(root).rglob('possession_metrics.py'))
    if found: pm_path=found[0]
    else:
        url = ('https://raw.githubusercontent.com/metrica-sports/'
               'more-than-a-hack-2027/main/evaluation/possession_metrics.py')
        print('Telechargement du scoreur officiel:',url)
        try:
            urllib.request.urlretrieve(url,str(pm_path))
        except Exception as exc:
            raise RuntimeError('Scoreur officiel non disponible. Active Kaggle Internet'
                ' ou ajoute evaluation/possession_metrics.py comme dataset. '+str(exc))
spec=importlib.util.spec_from_file_location('xc_official_pm',str(pm_path))
pm=importlib.util.module_from_spec(spec)
spec.loader.exec_module(pm)
assert tuple(pm.SPELL_IOUS)==(.3,.4,.5,.6,.7)

# Controle STRICT de l'exact fichier que nous allons envoyer.
submission = XC_O/'Xcelerate FC.csv'
if not submission.exists(): submission=XC_O/'Xcelerate.csv'
if not submission.exists():raise FileNotFoundError('Executer la derniere cellule de soumission')
table=pm.read_label_csv(str(submission))
reason=pm.validate_submission(table,199319)
if reason: raise RuntimeError('ECHEC VALIDATEUR OFFICIEL: '+reason)
print('VALIDATION OFFICIELLE DU FORMAT: OK')
print('Fichier:',submission,'| lignes:',len(table), '| taille:',submission.stat().st_size)
print('SHA256:',hashlib.sha256(submission.read_bytes()).hexdigest())

# Les cinq minutes publiques sont un CONTROLE, PAS un set d'entrainement ou de tuning.
roots=[Path('/kaggle/input'),XC_O]
found=[]
for root in roots:
    if root.exists():found.extend(root.rglob('BAR-ATH_possession_gt_public.csv'))
if not found:
    print('GT publique absente: ajouter BAR-DATA. Les controles de format restent OK.')
else:
    truth=pd.read_csv(found[0]);assert list(truth.columns)==['frame','label']
    assert len(truth)==199319
    assert np.array_equal(truth.frame.to_numpy(),np.arange(199319))
    gt=pm.label_codes(truth)
    assert int(np.sum(gt!=pm.X))==7500,'GT publique inattendue'
    print('GT publique:',found[0], '| 7500 frames uniquement')

    # Diagnostic non classe, ordre de declaration des modeles maintenu.
    rows=[]
    for name,p in TARGET_PREDS.items():
        arr=np.asarray(p)
        if len(arr)!=199319:raise ValueError('Longueur pred incorrecte '+name)
        codes=pm.label_codes(arr)
        result=pm.evaluate(gt,codes,bootstrap_n=0)
        pc=result['per_class']
        rows.append(dict(modele=name,est_modele_exporte=(name==selected),
            spellF1_officiel=result['spell_f1_mean'],
            macroF1_officiel=result['macro_f1'],
            F1_H=pc['H']['f1'],F1_A=pc['A']['f1'],F1_D=pc['D']['f1'],
            precision_D=pc['D']['precision'],rappel_D=pc['D']['recall'],
            possessions_pred=result['n_pred_spells'],
            possessions_GT=result['n_gt_spells'],
            fragmentation=result['fragmentation'],
            n_frames_GT=result['n_graded'],
            abstentions_X_sur_GT=result['n_abstain_on_graded'],
            transition_F1_25f=result['transitions'][25]['f1']))
    pub=pd.DataFrame(rows)
    print('\n===== COMPARAISON PUBLIQUE : DIAGNOSTIC NON CLASSE / NON SELECTIF =====')
    display(pub.round(4))
    # Garde l'ordre des modeles : aucun classement, aucun argmax, aucune selection.
    pub.to_csv(XC_O/'XC_PUBLIC_5MIN_DIAGNOSTIC_NON_SELECTIF.csv',index=False)
    print('\nCONFUSION 5 MIN : MODELE DEJA VERROUILLE',selected)
    import pandas as pd
    selected_pred=pm.label_codes(np.asarray(TARGET_PREDS[selected]))
    cm=pm.scored_confusion(pm.confusion(gt,selected_pred))
    confusion=pd.DataFrame(cm,index=['GT_H','GT_A','GT_D'],columns=['P_H','P_A','P_D','P_X'])
    display(confusion)
    r=pub.loc[pub.modele==selected].iloc[0]
    summary='\n'.join([
        'XCELERATE — DIAGNOSTIC PUBLIQUE APRES SELECTION EXTERNE',
        'Candidat verrouille: '+str(selected),
        'GT publique: 7500 frames / 17 possessions GT environ',
        f'spellF1 officiel={r.spellF1_officiel:.4f}',
        f'macroF1 officiel={r.macroF1_officiel:.4f}',
        f'F1_H={r.F1_H:.4f}; F1_A={r.F1_A:.4f}; F1_D={r.F1_D:.4f}',
        f'precision_D={r.precision_D:.4f}; rappel_D={r.rappel_D:.4f}',
        f'possessions_pred={int(r.possessions_pred)}',
        'Ces valeurs ne servent ni a classer, ni a choisir, ni a ajuster un modele.',
        '',pub.to_string(index=False,float_format=lambda x:f'{x:.4f}')])
    (XC_O/'XC_PUBLIC_RAPPORT_A_M_ENVOYER.txt').write_text(summary,encoding='utf-8')
    print('\n===== TEXTE A M ENVOYER =====\n'+summary)
    print('\nRapport public sauvegarde:',XC_O/'XC_PUBLIC_RAPPORT_A_M_ENVOYER.txt')
print('\nRAPPEL : la metrique de classement officielle est spellF1 sur le match prive entier.')


**Xcelerate FC | Chronologie des divergences et repérage vidéo**

Cette cellule lit les prédictions déjà calculées dans le notebook principal. Aucune vérité terrain n’est utilisée. Nous examinons les différences entre le candidat sélectionné et les autres approches, avec le numéro exact de frame, le temps de vidéo et l’horloge de jeu estimée à partir des coups d’envoi.

Les graphiques combinent les états H, A, D, X, la proximité du ballon et sa visibilité. Les CSV exportés sont destinés au débogage interne et ne doivent pas être mis en ligne, car ils dérivent du match privé.

Exécution : à la fin du notebook intégral, sans réinitialiser l’environnement Python.

In [ ]:
# XCELERATE FC | Cellule C : inspecter les divergences sur la chronologie video
# Exécuter après le notebook intégral, sans consulter la GT de la compétition.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
from IPython.display import display, Markdown

for req in ('TARGET_PREDS', 'FPS', 'HALVES', 'SECOND_KICKOFF', 'features'):
    if req not in globals():
        raise RuntimeError(f'Le notebook integral doit etre execute avant cette cellule : {req} absent.')

XC_C_OUT = OUT if 'OUT' in globals() else Path('/kaggle/working')
XC_C_OUT = Path(XC_C_OUT)
XC_C_OUT.mkdir(parents=True, exist_ok=True)
XC_C_PRED = {k:np.asarray(v).astype(str) for k,v in TARGET_PREDS.items()}
XC_C_LEN = len(next(iter(XC_C_PRED.values())))
assert all(len(v)==XC_C_LEN for v in XC_C_PRED.values())
XC_C_MASK = (MASK_FLEX.copy() if 'MASK_FLEX' in globals() else np.zeros(XC_C_LEN,dtype=bool))
if not XC_C_MASK.any():
    for a,b in HALVES: XC_C_MASK[a:b+1]=True
    XC_C_MASK[HALVES[1][0]:SECOND_KICKOFF]=False

# Parametres de consultation: changer sans toucher aux predictions.
XC_C_REFERENCE = SELECTED if 'SELECTED' in globals() and SELECTED in XC_C_PRED else 'fusion_control_multiscale'
if XC_C_REFERENCE not in XC_C_PRED:
    XC_C_REFERENCE = list(XC_C_PRED)[0]
XC_C_MIN_DURATION_S = 1.0
XC_C_MAX_PLOT_S = 32.0
XC_C_TOP_N = 35
XC_C_SKIP_NEAREST = True
XC_C_PLOT_NAMES = [x for x in [
    XC_C_REFERENCE,'switch_strong','baseline_hmm','control_strong',
    'fusion_D_V2_strong','D_V2_strong','adaptive_multiscale','temperature3_strong',
    'control_restart_strong','TabICLv2_fusion'
] if x in XC_C_PRED]
XC_C_PLOT_NAMES = list(dict.fromkeys(XC_C_PLOT_NAMES))

def xc_timecode(sec):
    sec=float(sec)
    h=int(sec//3600);m=int((sec%3600)//60);s=sec%60
    return f'{h:02d}:{m:02d}:{s:05.2f}' if h else f'{m:02d}:{s:05.2f}'

def xc_match_clock(frame):
    frame=int(frame)
    if frame<HALVES[0][0]:return 'before graded start'
    if frame<=HALVES[0][1]:return '1H '+xc_timecode((frame-HALVES[0][0])/FPS)
    if frame<SECOND_KICKOFF:return 'halftime/pre-kickoff'
    if frame<=HALVES[1][1]:return '2H '+xc_timecode(45*60+(frame-SECOND_KICKOFF)/FPS)
    return 'after graded end'

def xc_events(ref_name, compare_name):
    base=XC_C_PRED[ref_name];other=XC_C_PRED[compare_name]
    active=XC_C_MASK & (base != other)
    cuts=np.r_[0,np.flatnonzero((active[1:]!=active[:-1])|
               (base[1:]!=base[:-1])|(other[1:]!=other[:-1]))+1,len(base)]
    rows=[]
    for a,b in zip(cuts[:-1],cuts[1:]):
        if not active[a] or (b-a)/FPS<XC_C_MIN_DURATION_S:continue
        rows.append({'reference':ref_name,'autre_modele':compare_name,'frame_debut':int(a),
                     'frame_fin':int(b-1),'n_frames':int(b-a),'duree_s':round((b-a)/FPS,2),
                     'video_debut':xc_timecode(a/FPS),'video_fin':xc_timecode((b-1)/FPS),
                     'temps_jeu':xc_match_clock(a),
                     'reference_label':base[a],'autre_label':other[a],
                     'type_divergence':f'{base[a]} -> {other[a]}'})
    return rows

XC_C_ROWS=[]
for name in XC_C_PRED:
    if name==XC_C_REFERENCE or (XC_C_SKIP_NEAREST and name=='nearest'):continue
    XC_C_ROWS.extend(xc_events(XC_C_REFERENCE,name))
XC_C_LONG=pd.DataFrame(XC_C_ROWS)
if XC_C_LONG.empty:raise RuntimeError('Aucune divergence trouvee pour cette configuration.')
XC_C_LONG=XC_C_LONG.sort_values('duree_s',ascending=False).reset_index(drop=True)
XC_C_LONG.insert(0,'rang',np.arange(1,len(XC_C_LONG)+1))
XC_C_LONG.to_csv(XC_C_OUT/'XC_C_divergences_longues.csv',index=False)

XC_C_SHARE=[]
for name, v in XC_C_PRED.items():
    if name==XC_C_REFERENCE:continue
    changed=XC_C_MASK&(v!=XC_C_PRED[XC_C_REFERENCE])
    XC_C_SHARE.append({'modele':name,'frames_divergentes':int(changed.sum()),
                       'desaccord_pct':round(100*changed.sum()/XC_C_MASK.sum(),3),
                       'plus_long_desaccord_s':float(XC_C_LONG.loc[XC_C_LONG.autre_modele.eq(name),'duree_s'].max()) if name in XC_C_LONG.autre_modele.values else 0})
XC_C_SHARE=pd.DataFrame(XC_C_SHARE).sort_values('desaccord_pct',ascending=False)
XC_C_SHARE.to_csv(XC_C_OUT/'XC_C_comparaison_modeles.csv',index=False)
print('REFERENCE UTILISEE :',XC_C_REFERENCE)
print('Divergences >=',XC_C_MIN_DURATION_S,'seconde(s) | seulement les frames evaluables sans D impose')
print('\n=== DESACCORD GLOBAL PAR MODELE ===')
display(XC_C_SHARE.round(2))
print('\n=== TOP',XC_C_TOP_N,'SEQUENCES LES PLUS LONGUES ===')
display(XC_C_LONG.head(XC_C_TOP_N))
print('\n=== PLUS LONG DESACCORD PAR TYPE ===')
display(XC_C_LONG.sort_values('duree_s',ascending=False).drop_duplicates('type_divergence')
        [['type_divergence','autre_modele','frame_debut','frame_fin','duree_s','video_debut','video_fin','temps_jeu']])

# Echantillon exact des frames ou au moins une autre famille differe du candidat.
XC_C_OTHER=[k for k in XC_C_PLOT_NAMES if k!=XC_C_REFERENCE]
XC_C_ANY=XC_C_MASK&np.any([XC_C_PRED[k]!=XC_C_PRED[XC_C_REFERENCE] for k in XC_C_OTHER],axis=0)
XC_C_IDX=np.flatnonzero(XC_C_ANY)
XC_C_FRAME_TBL=pd.DataFrame({'frame':XC_C_IDX,
        'timecode':[xc_timecode(f/FPS) for f in XC_C_IDX],
        'match_clock':[xc_match_clock(f) for f in XC_C_IDX]})
for name in XC_C_PLOT_NAMES:
    XC_C_FRAME_TBL[name]=XC_C_PRED[name][XC_C_IDX]
XC_C_FRAME_TBL.to_csv(XC_C_OUT/'XC_C_frames_desaccord_detail.csv',index=False)

# Visualization reusable pour n'importe quel intervalle, en frames.
xc_colors=ListedColormap(['#2867B2','#EA941D','#8B929C','#333B49'])
xc_norm=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)
xc_encoder={'H':0,'A':1,'D':2,'X':3}

def xc_plot_interval(frame_debut, duree_s=30., noms=None):
    a=int(np.clip(frame_debut,0,XC_C_LEN-1)); b=min(XC_C_LEN,a+max(2,int(round(duree_s*FPS))))
    names=XC_C_PLOT_NAMES if noms is None else [k for k in noms if k in XC_C_PRED]
    if not names:raise ValueError('Aucun modele valide')
    M=np.array([[xc_encoder.get(str(x),3) for x in XC_C_PRED[k][a:b]] for k in names])
    t=np.arange(a,b)/FPS
    fig,axs=plt.subplots(3,1,figsize=(16,8),sharex=True,
           gridspec_kw={'height_ratios':[4,1.1,.8]})
    axs[0].imshow(M,aspect='auto',interpolation='nearest',cmap=xc_colors,norm=xc_norm,
                  extent=(t[0],t[-1]+1/FPS,len(names)-.5,-.5))
    axs[0].set_yticks(np.arange(len(names)),names)
    axs[0].set_title(f'Predictions frame {a} a {b-1} | video {xc_timecode(a/FPS)} a {xc_timecode((b-1)/FPS)} | {xc_match_clock(a)}',loc='left')
    axs[0].grid(axis='x',alpha=.15)
    for k,col in [('nearest_dist_diff','#2563EB'),('closest_distance','#D97706')]:
        if k in features.columns:
            arr=pd.to_numeric(features[k],errors='coerce').to_numpy(float)[a:b]
            axs[1].plot(t,arr,label=k,color=col,linewidth=.85)
    axs[1].axhline(0,color='grey',linewidth=.6)
    axs[1].legend(loc='upper right',fontsize=8)
    axs[1].set_ylabel('Distances (m)')
    if 'ball_visible' in features.columns:
        vis=pd.to_numeric(features.ball_visible,errors='coerce').to_numpy(float)[a:b]
        axs[2].step(t,vis,where='post',color='#16845D',label='ball_visible')
    if 'ball_gap_quality' in features.columns:
        quality=pd.to_numeric(features.ball_gap_quality,errors='coerce').to_numpy(float)[a:b]
        axs[2].plot(t,quality,color='#B15E26',linewidth=1,label='ball_gap_quality')
    axs[2].set_ylim(-.12,1.15);axs[2].set_yticks([0,1]);axs[2].set_ylabel('Visibilite')
    axs[2].set_xlabel('Temps dans la video (secondes)')
    axs[2].legend(loc='upper right',fontsize=8)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return a,b

# Zoom sur les 4 plus longues divergences (hors nearest) :
for _,item in XC_C_LONG.head(4).iterrows():
    print('\nINTERVALLE',item.rang,'|',item.autre_modele,'|',item.type_divergence,'|',item.duree_s,'s')
    a,b=xc_plot_interval(max(0,item.frame_debut-round(2*FPS)),min(XC_C_MAX_PLOT_S,item.duree_s+4))

print('\n=== UTILISATION MANUELLE ===')
print("xc_plot_interval(158366, 24)   # exemple, 24 secondes a partir de la frame 158366")
print("xc_plot_interval(11185, 30, ['fusion_control_multiscale', 'switch_strong','fusion_D_V2_strong'])")
print('\nFichiers disponibles dans',XC_C_OUT,':')
print('XC_C_divergences_longues.csv | XC_C_comparaison_modeles.csv | XC_C_frames_desaccord_detail.csv')
print('Ne pas publier les frames derivees du match prive sur GitHub. Usage interne uniquement.')
